In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash kailash-ml kailash-kaizen polars plotly gdown python-dotenv nest-asyncio

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP M5 inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated from shared/. Click the ▼ arrow on the left to hide.
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations


# ── kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model

# ── diagnostics.py ──
"""DL Diagnostics Toolkit — clinical instruments for deep-network training.

Wraps PyTorch forward/backward hooks into a student-friendly API that
surfaces the four failure modes professionals must recognise:

    1. Stethoscope  — loss-curve shape (under/over-fit, instability)
    2. Blood Test   — gradient flow per layer (vanishing / exploding)
    3. X-Ray        — activation statistics & dead neurons
    4. Prescription — automated diagnosis with actionable suggestions

Typical use inside an exercise::


    with DLDiagnostics(model) as diag:
        diag.track_gradients()
        diag.track_activations()
        diag.track_dead_neurons()
        for epoch in range(epochs):
            for batch in dataloader:
                loss = train_step(model, batch)
                diag.record_batch(loss=loss.item(),
                                  lr=optimizer.param_groups[0]["lr"])
            diag.record_epoch(val_loss=evaluate(model, val_loader))
        diag.plot_training_dashboard().show()
        diag.report()

All DataFrames are Polars. All plots are Plotly. No matplotlib, no pandas.
"""

import logging
import math
from collections.abc import Callable
from dataclasses import dataclass, field
from typing import Any

import numpy as np
import plotly.graph_objects as go
import polars as pl
import torch
import torch.nn as nn
from plotly.subplots import make_subplots
from torch.utils.data import DataLoader


logger = logging.getLogger(__name__)

# Module names whose outputs are "dead-neuron sensitive" — we track fraction
# of zero outputs for these specifically.
_DEAD_NEURON_SENSITIVE: tuple[type[nn.Module], ...] = (
    nn.ReLU,
    nn.LeakyReLU,
    nn.GELU,
    nn.ELU,
    nn.SiLU,
)

# Module names whose outputs we monitor for activation statistics.
_ACTIVATION_MONITORED: tuple[type[nn.Module], ...] = (
    nn.Linear,
    nn.Conv1d,
    nn.Conv2d,
    nn.Conv3d,
    nn.ReLU,
    nn.LeakyReLU,
    nn.GELU,
    nn.ELU,
    nn.SiLU,
    nn.Tanh,
    nn.Sigmoid,
    nn.BatchNorm1d,
    nn.BatchNorm2d,
    nn.LayerNorm,
)


@dataclass
class _HookHandles:
    """Container for registered hook handles so we can detach cleanly."""

    gradient: list[torch.utils.hooks.RemovableHandle] = field(default_factory=list)
    activation: list[torch.utils.hooks.RemovableHandle] = field(default_factory=list)
    dead_neuron: list[torch.utils.hooks.RemovableHandle] = field(default_factory=list)
    grad_cam: list[torch.utils.hooks.RemovableHandle] = field(default_factory=list)

    def all(self) -> list[torch.utils.hooks.RemovableHandle]:
        return self.gradient + self.activation + self.dead_neuron + self.grad_cam


class DLDiagnostics:
    """Clinical instrumentation for PyTorch training loops.

    Collects per-batch time series of gradient norms, activation statistics,
    dead-neuron fractions, and losses; exposes Plotly visualisations and an
    automated report that surfaces overfitting, vanishing gradients, and
    pathological dead-ReLU layers.

    Args:
        model: The ``nn.Module`` to instrument. The model is NOT modified;
            only forward/backward hooks are attached.
        dead_neuron_threshold: Fraction of zero outputs above which a layer
            is flagged as "dead" in :meth:`report`. Defaults to ``0.5``.
        window: Number of recent batches used for dead-neuron statistics.
            Defaults to ``64``.

    Example:
        >>> model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 1))
        >>> with DLDiagnostics(model) as diag:
        ...     diag.track_gradients()
        ...     diag.track_activations()
        ...     # ... training loop ...
    """

    def __init__(
        self,
        model: nn.Module,
        *,
        dead_neuron_threshold: float = 0.5,
        window: int = 64,
    ) -> None:
        if not isinstance(model, nn.Module):
            raise TypeError(
                f"DLDiagnostics requires an nn.Module; got {type(model).__name__}"
            )
        if not 0.0 < dead_neuron_threshold < 1.0:
            raise ValueError("dead_neuron_threshold must be in (0, 1)")
        if window < 1:
            raise ValueError("window must be >= 1")

        self.model = model
        self.device = get_device()
        self.dead_neuron_threshold = dead_neuron_threshold
        self.window = window

        # Time series storage — lists of dicts, converted to Polars on demand.
        self._grad_log: list[dict[str, Any]] = []
        self._act_log: list[dict[str, Any]] = []
        self._dead_log: list[dict[str, Any]] = []
        self._batch_log: list[dict[str, Any]] = []
        self._epoch_log: list[dict[str, Any]] = []

        # Running per-layer firing masks for dead-neuron detection.
        # Key: layer name -> tensor of firing counts per neuron (1D).
        self._firing_counts: dict[str, torch.Tensor] = {}
        self._firing_samples: dict[str, int] = {}

        # Counters bound to hook closures so they share scope.
        self._batch_idx = 0
        self._epoch_idx = 0

        self._handles = _HookHandles()
        self._tracking = {"gradients": False, "activations": False, "dead": False}

        # Grad-CAM capture buffers (populated on demand).
        self._gradcam_activation: torch.Tensor | None = None
        self._gradcam_gradient: torch.Tensor | None = None

        logger.info(
            "dldiagnostics.init",
            extra={
                "model_class": type(model).__name__,
                "device": str(self.device),
                "window": window,
            },
        )

    # ── Context-manager support ────────────────────────────────────────────

    def __enter__(self) -> DLDiagnostics:
        return self

    def __exit__(self, exc_type, exc, tb) -> None:  # noqa: D401
        self.detach()

    def __del__(self) -> None:  # pragma: no cover - best-effort cleanup
        try:
            self.detach()
        except Exception:
            # Finalizers MUST NOT raise. Silent cleanup is the documented
            # contract for __del__ in rules/patterns.md.
            pass

    # ── Hook registration ──────────────────────────────────────────────────

    def track_gradients(self) -> DLDiagnostics:
        """Register backward hooks on every trainable parameter.

        Records the L2 norm of each parameter's gradient at every backward
        pass, keyed by parameter name.

        Returns:
            ``self`` for chaining.
        """
        if self._tracking["gradients"]:
            return self
        for name, param in self.model.named_parameters():
            if not param.requires_grad:
                continue
            handle = param.register_hook(self._make_grad_hook(name))
            self._handles.gradient.append(handle)
        self._tracking["gradients"] = True
        logger.info(
            "dldiagnostics.track_gradients",
            extra={"hooks_registered": len(self._handles.gradient)},
        )
        return self

    def track_activations(self) -> DLDiagnostics:
        """Register forward hooks on monitored submodules.

        Records mean/std/min/max/dead-fraction of activations per layer
        at every forward pass.

        Returns:
            ``self`` for chaining.
        """
        if self._tracking["activations"]:
            return self
        for name, module in self.model.named_modules():
            if name == "" or not isinstance(module, _ACTIVATION_MONITORED):
                continue
            handle = module.register_forward_hook(self._make_act_hook(name))
            self._handles.activation.append(handle)
        self._tracking["activations"] = True
        logger.info(
            "dldiagnostics.track_activations",
            extra={"hooks_registered": len(self._handles.activation)},
        )
        return self

    def track_dead_neurons(self) -> DLDiagnostics:
        """Register forward hooks on ReLU-family layers to track dead neurons.

        A "neuron" here is a channel (Conv) or output unit (Linear). The
        rolling fraction of batches where that neuron output zero is tracked.

        Returns:
            ``self`` for chaining.
        """
        if self._tracking["dead"]:
            return self
        for name, module in self.model.named_modules():
            if name == "" or not isinstance(module, _DEAD_NEURON_SENSITIVE):
                continue
            handle = module.register_forward_hook(self._make_dead_hook(name))
            self._handles.dead_neuron.append(handle)
        self._tracking["dead"] = True
        logger.info(
            "dldiagnostics.track_dead_neurons",
            extra={"hooks_registered": len(self._handles.dead_neuron)},
        )
        return self

    def detach(self) -> None:
        """Remove ALL registered hooks and release references.

        Safe to call multiple times. Invoked automatically on context exit.
        """
        for handle in self._handles.all():
            try:
                handle.remove()
            except Exception:
                # Hook removal failures are benign (module may already be
                # gone). See rules/zero-tolerance.md Rule 3 carve-out for
                # cleanup paths.
                pass
        self._handles = _HookHandles()
        self._tracking = {k: False for k in self._tracking}
        self._gradcam_activation = None
        self._gradcam_gradient = None

    # ── Recording ─────────────────────────────────────────────────────────

    def record_batch(self, *, loss: float, lr: float | None = None) -> None:
        """Record per-batch scalar training signals.

        Args:
            loss: Training loss for the batch (post-backward).
            lr: Current learning rate (optional; read from optimizer).
        """
        if not math.isfinite(loss):
            logger.warning(
                "dldiagnostics.record_batch.nonfinite_loss",
                extra={"loss": str(loss), "batch": self._batch_idx},
            )
        self._batch_log.append(
            {
                "batch": self._batch_idx,
                "epoch": self._epoch_idx,
                "loss": float(loss),
                "lr": float(lr) if lr is not None else float("nan"),
            }
        )
        self._batch_idx += 1

    def record_epoch(
        self,
        *,
        val_loss: float | None = None,
        train_loss: float | None = None,
        **extra: float,
    ) -> None:
        """Record per-epoch summary metrics.

        Args:
            val_loss: Validation loss at epoch end.
            train_loss: Mean training loss for the epoch. If ``None``, it is
                computed from the batches in this epoch.
            **extra: Any additional scalar metrics to persist.
        """
        if train_loss is None:
            epoch_batches = [
                b for b in self._batch_log if b["epoch"] == self._epoch_idx
            ]
            if epoch_batches:
                train_loss = float(np.mean([b["loss"] for b in epoch_batches]))
        entry = {
            "epoch": self._epoch_idx,
            "train_loss": train_loss if train_loss is not None else float("nan"),
            "val_loss": val_loss if val_loss is not None else float("nan"),
            **{k: float(v) for k, v in extra.items()},
        }
        self._epoch_log.append(entry)
        self._epoch_idx += 1

    # ── Public DataFrame accessors ────────────────────────────────────────

    def gradients_df(self) -> pl.DataFrame:
        """Polars DataFrame of per-layer gradient norms by batch."""
        if not self._grad_log:
            return pl.DataFrame(
                schema={
                    "batch": pl.Int64,
                    "layer": pl.Utf8,
                    "grad_norm": pl.Float64,
                    "grad_rms": pl.Float64,
                    "update_ratio": pl.Float64,
                }
            )
        return pl.DataFrame(self._grad_log)

    def activations_df(self) -> pl.DataFrame:
        """Polars DataFrame of per-layer activation statistics by batch."""
        if not self._act_log:
            return pl.DataFrame(
                schema={
                    "batch": pl.Int64,
                    "layer": pl.Utf8,
                    "act_kind": pl.Utf8,
                    "mean": pl.Float64,
                    "std": pl.Float64,
                    "min": pl.Float64,
                    "max": pl.Float64,
                    "dead_fraction": pl.Float64,
                    "inactivity_fraction": pl.Float64,
                }
            )
        return pl.DataFrame(self._act_log)

    def dead_neurons_df(self) -> pl.DataFrame:
        """Polars DataFrame of current per-layer dead-neuron fractions."""
        rows: list[dict[str, Any]] = []
        for name, counts in self._firing_counts.items():
            n_samples = max(self._firing_samples.get(name, 1), 1)
            dead_mask = (counts / n_samples) < 1e-6
            rows.append(
                {
                    "layer": name,
                    "n_neurons": int(counts.numel()),
                    "n_dead": int(dead_mask.sum().item()),
                    "dead_fraction": float(dead_mask.float().mean().item()),
                }
            )
        if not rows:
            return pl.DataFrame(
                schema={
                    "layer": pl.Utf8,
                    "n_neurons": pl.Int64,
                    "n_dead": pl.Int64,
                    "dead_fraction": pl.Float64,
                }
            )
        return pl.DataFrame(rows)

    def batches_df(self) -> pl.DataFrame:
        """Polars DataFrame of per-batch scalars (loss, lr)."""
        if not self._batch_log:
            return pl.DataFrame(
                schema={
                    "batch": pl.Int64,
                    "epoch": pl.Int64,
                    "loss": pl.Float64,
                    "lr": pl.Float64,
                }
            )
        return pl.DataFrame(self._batch_log)

    def epochs_df(self) -> pl.DataFrame:
        """Polars DataFrame of per-epoch summary metrics."""
        if not self._epoch_log:
            return pl.DataFrame(
                schema={
                    "epoch": pl.Int64,
                    "train_loss": pl.Float64,
                    "val_loss": pl.Float64,
                }
            )
        return pl.DataFrame(self._epoch_log)

    # ── Plots ─────────────────────────────────────────────────────────────

    def plot_loss_curves(self) -> go.Figure:
        """Loss stethoscope: train vs val curves with overfitting callout.

        Returns:
            A Plotly Figure with two traces (train / val) and annotations
            for detected overfitting epoch (if any).
        """
        epochs = self.epochs_df()
        batches = self.batches_df()
        fig = go.Figure()
        if batches.height:
            fig.add_trace(
                go.Scatter(
                    x=batches["batch"].to_list(),
                    y=batches["loss"].to_list(),
                    mode="lines",
                    name="train (per batch)",
                    line=dict(color="lightblue", width=1),
                    opacity=0.5,
                )
            )
        if epochs.height:
            fig.add_trace(
                go.Scatter(
                    x=epochs["epoch"].to_list(),
                    y=epochs["train_loss"].to_list(),
                    mode="lines+markers",
                    name="train (epoch mean)",
                    line=dict(color="steelblue", width=2),
                )
            )
            if epochs["val_loss"].is_not_null().any():
                fig.add_trace(
                    go.Scatter(
                        x=epochs["epoch"].to_list(),
                        y=epochs["val_loss"].to_list(),
                        mode="lines+markers",
                        name="val",
                        line=dict(color="firebrick", width=2),
                    )
                )
        overfit_epoch = self._detect_overfit_epoch()
        if overfit_epoch is not None:
            fig.add_vline(
                x=overfit_epoch,
                line=dict(color="orange", dash="dash"),
                annotation_text=f"overfitting suspected @ epoch {overfit_epoch}",
                annotation_position="top",
            )
        fig.update_layout(
            title="Loss Curves (Stethoscope)",
            xaxis_title="step",
            yaxis_title="loss",
            template="plotly_white",
            hovermode="x unified",
        )
        return fig

    def plot_gradient_flow(self) -> go.Figure:
        """Blood test: per-layer gradient norm over time.

        One trace per tracked parameter. Layers whose gradient norm collapses
        toward zero are the vanishing-gradient culprits.
        """
        df = self.gradients_df()
        fig = go.Figure()
        if df.height == 0:
            fig.update_layout(
                title="Gradient Flow (Blood Test) — no data",
                template="plotly_white",
            )
            return fig
        for layer in df["layer"].unique().to_list():
            sub = df.filter(pl.col("layer") == layer)
            fig.add_trace(
                go.Scatter(
                    x=sub["batch"].to_list(),
                    y=sub["grad_norm"].to_list(),
                    mode="lines",
                    name=layer,
                    hovertemplate=f"{layer}<br>batch=%{{x}}<br>‖∇‖=%{{y:.3e}}",
                )
            )
        fig.update_layout(
            title="Gradient Flow by Layer (Blood Test)",
            xaxis_title="batch",
            yaxis_title="gradient L2 norm",
            yaxis_type="log",
            template="plotly_white",
        )
        return fig

    def plot_activation_stats(self) -> go.Figure:
        """X-Ray: activation mean ± std per layer over time."""
        df = self.activations_df()
        fig = go.Figure()
        if df.height == 0:
            fig.update_layout(
                title="Activation Statistics (X-Ray) — no data",
                template="plotly_white",
            )
            return fig
        for layer in df["layer"].unique().to_list():
            sub = df.filter(pl.col("layer") == layer)
            fig.add_trace(
                go.Scatter(
                    x=sub["batch"].to_list(),
                    y=sub["mean"].to_list(),
                    mode="lines",
                    name=f"{layer} — mean",
                    hovertemplate=(
                        f"{layer}<br>batch=%{{x}}<br>mean=%{{y:.3f}}<br>"
                        "std=%{customdata:.3f}"
                    ),
                    customdata=sub["std"].to_list(),
                )
            )
        fig.update_layout(
            title="Activation Mean per Layer (X-Ray)",
            xaxis_title="batch",
            yaxis_title="activation mean",
            template="plotly_white",
        )
        return fig

    def plot_dead_neurons(self) -> go.Figure:
        """X-Ray: fraction of dead neurons per ReLU-family layer."""
        df = self.dead_neurons_df()
        fig = go.Figure()
        if df.height == 0:
            fig.update_layout(
                title="Dead Neurons (X-Ray) — no ReLU-family layers tracked",
                template="plotly_white",
            )
            return fig
        colors = [
            "firebrick" if frac > self.dead_neuron_threshold else "steelblue"
            for frac in df["dead_fraction"].to_list()
        ]
        fig.add_trace(
            go.Bar(
                x=df["layer"].to_list(),
                y=df["dead_fraction"].to_list(),
                marker_color=colors,
                text=[
                    f"{int(n)}/{int(t)}" for n, t in zip(df["n_dead"], df["n_neurons"])
                ],
                textposition="outside",
            )
        )
        fig.add_hline(
            y=self.dead_neuron_threshold,
            line=dict(color="orange", dash="dash"),
            annotation_text=f"alert threshold ({self.dead_neuron_threshold:.0%})",
        )
        fig.update_layout(
            title="Dead Neuron Fraction by Layer (X-Ray)",
            xaxis_title="layer",
            yaxis_title="fraction dead",
            yaxis=dict(range=[0, 1]),
            template="plotly_white",
            showlegend=False,
        )
        return fig

    def plot_training_dashboard(self) -> go.Figure:
        """Vital signs: 2x2 dashboard (loss, grad flow, activations, LR)."""
        fig = make_subplots(
            rows=2,
            cols=2,
            subplot_titles=(
                "Loss (Stethoscope)",
                "Gradient Flow (Blood Test)",
                "Activation Mean (X-Ray)",
                "Learning Rate",
            ),
        )

        # (1,1) Loss
        epochs = self.epochs_df()
        batches = self.batches_df()
        if batches.height:
            fig.add_trace(
                go.Scatter(
                    x=batches["batch"].to_list(),
                    y=batches["loss"].to_list(),
                    mode="lines",
                    name="train loss",
                    line=dict(color="steelblue", width=1),
                ),
                row=1,
                col=1,
            )
        if epochs.height and epochs["val_loss"].is_not_null().any():
            # Place val loss at the last batch of each epoch for alignment.
            val_x = []
            for ep in epochs["epoch"].to_list():
                ep_batches = batches.filter(pl.col("epoch") == ep)
                val_x.append(
                    int(ep_batches["batch"].max()) if ep_batches.height else ep
                )
            fig.add_trace(
                go.Scatter(
                    x=val_x,
                    y=epochs["val_loss"].to_list(),
                    mode="lines+markers",
                    name="val loss",
                    line=dict(color="firebrick"),
                ),
                row=1,
                col=1,
            )

        # (1,2) Gradient flow
        gdf = self.gradients_df()
        if gdf.height:
            for layer in gdf["layer"].unique().to_list():
                sub = gdf.filter(pl.col("layer") == layer)
                fig.add_trace(
                    go.Scatter(
                        x=sub["batch"].to_list(),
                        y=sub["grad_norm"].to_list(),
                        mode="lines",
                        name=layer,
                        showlegend=False,
                    ),
                    row=1,
                    col=2,
                )
            fig.update_yaxes(type="log", row=1, col=2)

        # (2,1) Activation mean
        adf = self.activations_df()
        if adf.height:
            for layer in adf["layer"].unique().to_list():
                sub = adf.filter(pl.col("layer") == layer)
                fig.add_trace(
                    go.Scatter(
                        x=sub["batch"].to_list(),
                        y=sub["mean"].to_list(),
                        mode="lines",
                        name=layer,
                        showlegend=False,
                    ),
                    row=2,
                    col=1,
                )

        # (2,2) LR
        if batches.height and batches["lr"].is_not_null().any():
            fig.add_trace(
                go.Scatter(
                    x=batches["batch"].to_list(),
                    y=batches["lr"].to_list(),
                    mode="lines",
                    name="lr",
                    line=dict(color="darkgreen"),
                    showlegend=False,
                ),
                row=2,
                col=2,
            )

        fig.update_layout(
            title="Training Dashboard (Vital Signs)",
            template="plotly_white",
            height=720,
        )
        return fig

    def plot_lr_vs_loss(self) -> go.Figure:
        """Plot LR vs loss (useful after an :meth:`lr_range_test`)."""
        df = self.batches_df()
        fig = go.Figure()
        if df.height == 0 or df["lr"].is_null().all():
            fig.update_layout(title="LR vs Loss — no data", template="plotly_white")
            return fig
        fig.add_trace(
            go.Scatter(
                x=df["lr"].to_list(),
                y=df["loss"].to_list(),
                mode="lines",
                line=dict(color="steelblue"),
            )
        )
        fig.update_layout(
            title="Learning Rate vs Loss",
            xaxis_title="learning rate",
            yaxis_title="loss",
            xaxis_type="log",
            template="plotly_white",
        )
        return fig

    def plot_weight_distributions(self) -> go.Figure:
        """Histogram of weight values, one trace per parameter tensor."""
        fig = go.Figure()
        for name, param in self.model.named_parameters():
            if not param.requires_grad or param.ndim == 0:
                continue
            values = param.detach().cpu().flatten().numpy()
            fig.add_trace(go.Histogram(x=values, name=name, opacity=0.6))
        fig.update_layout(
            title="Weight Distributions",
            xaxis_title="value",
            yaxis_title="count",
            barmode="overlay",
            template="plotly_white",
        )
        return fig

    def plot_gradient_norms(self) -> go.Figure:
        """Mean gradient norm per layer across the run (bar chart)."""
        df = self.gradients_df()
        fig = go.Figure()
        if df.height == 0:
            fig.update_layout(title="Gradient Norms — no data", template="plotly_white")
            return fig
        summary = df.group_by("layer").agg(
            pl.col("grad_norm").mean().alias("mean_grad")
        )
        summary = summary.sort("mean_grad")
        fig.add_trace(
            go.Bar(
                x=summary["layer"].to_list(),
                y=summary["mean_grad"].to_list(),
                marker_color="steelblue",
            )
        )
        fig.update_layout(
            title="Mean Gradient Norm per Layer",
            xaxis_title="layer",
            yaxis_title="mean ‖∇‖",
            yaxis_type="log",
            template="plotly_white",
        )
        return fig

    # ── Automated report ──────────────────────────────────────────────────

    def report(self) -> dict[str, Any]:
        """Print a human-readable diagnosis and return findings as a dict.

        Findings covered:
            * Gradient flow (vanishing / exploding / healthy)
            * Dead neurons (per-layer ReLU-family)
            * Loss trend (overfitting, underfitting, instability)

        Returns:
            A dict with keys ``gradient_flow``, ``dead_neurons``, ``loss_trend``
            each mapping to a dict with ``severity`` and ``message``.
        """
        findings: dict[str, Any] = {}

        # 1. Gradient flow — uses SCALE-INVARIANT per-element RMS (grad_rms)
        # and update_ratio (‖∇W‖/‖W‖), matching slide 5F thresholds.
        gdf = self.gradients_df()
        if gdf.height and "grad_rms" in gdf.columns:
            stats = gdf.group_by("layer").agg(
                pl.col("grad_rms").mean().alias("rms"),
                pl.col("update_ratio").mean().alias("ur"),
            )
            min_rms_raw = stats["rms"].min()
            max_rms_raw = stats["rms"].max()
            min_ur_raw = stats["ur"].min()
            max_ur_raw = stats["ur"].max()
            min_rms = (
                float(min_rms_raw) if isinstance(min_rms_raw, (int, float)) else None
            )
            max_rms = (
                float(max_rms_raw) if isinstance(max_rms_raw, (int, float)) else None
            )
            min_ur = float(min_ur_raw) if isinstance(min_ur_raw, (int, float)) else 0.0
            max_ur = float(max_ur_raw) if isinstance(max_ur_raw, (int, float)) else 0.0
            if min_rms is None or max_rms is None or min_rms == 0:
                severity = "UNKNOWN"
                message = "Insufficient gradient data."
            elif min_rms < 1e-5 or min_ur < 1e-4:
                # Vanishing: RMS < 1e-5 OR update ratio < 1e-4 (matches slide 5F).
                worst_layer = (
                    stats.sort("rms").row(0, named=True)["layer"]
                    if min_rms < 1e-5
                    else stats.sort("ur").row(0, named=True)["layer"]
                )
                severity = "CRITICAL"
                message = (
                    f"Vanishing gradients at '{worst_layer}' — "
                    f"min RMS = {min_rms:.2e}, min update_ratio = {min_ur:.2e}. "
                    "Fix: verify pre-norm layout (LayerNorm/RMSNorm before block), "
                    "add residual connections, switch to GELU/SiLU, or use Kaiming init."
                )
            elif max_rms > 1e-2 or max_ur > 0.1:
                # Exploding: RMS > 1e-2 OR update ratio > 0.1 (matches slide 5F).
                worst_layer = (
                    stats.sort("rms", descending=True).row(0, named=True)["layer"]
                    if max_rms > 1e-2
                    else stats.sort("ur", descending=True).row(0, named=True)["layer"]
                )
                severity = "CRITICAL"
                message = (
                    f"Exploding gradients at '{worst_layer}' — "
                    f"max RMS = {max_rms:.2e}, max update_ratio = {max_ur:.2e}. "
                    "Fix: add gradient clipping (or adaptive: ZClip/AGC), reduce LR, "
                    "verify initialization (Kaiming for ReLU, Xavier for Tanh)."
                )
            elif max_rms / max(min_rms, 1e-12) > 1e3:
                severity = "WARNING"
                message = (
                    f"Large RMS spread across layers (max/min = "
                    f"{max_rms / min_rms:.1e}). Deep layers may be learning unevenly."
                )
            else:
                severity = "HEALTHY"
                message = (
                    f"Gradient flow OK (RMS range {min_rms:.2e}–{max_rms:.2e}, "
                    f"update_ratio range {min_ur:.2e}–{max_ur:.2e})."
                )
            findings["gradient_flow"] = {"severity": severity, "message": message}
        elif gdf.height:
            # Legacy path for dataframes without the new columns.
            findings["gradient_flow"] = {
                "severity": "UNKNOWN",
                "message": (
                    "grad_rms field missing — re-run with the current library "
                    "version to get scale-invariant diagnosis."
                ),
            }
        else:
            findings["gradient_flow"] = {
                "severity": "UNKNOWN",
                "message": "No gradient tracking enabled — call track_gradients().",
            }

        # 2. Dead neurons / saturation — uses ACTIVATION-TYPE-AWARE
        # inactivity_fraction from _act_log (ReLU: |x|<eps; Tanh: |x|>0.99;
        # Sigmoid: |x|>0.99 or |x|<0.01). This catches near-dead ReLU channels
        # that the legacy exact-zero check misses post-BatchNorm.
        adf = self.activations_df()
        if adf.height and "inactivity_fraction" in adf.columns:
            # Aggregate mean inactivity per layer (averaged across batches).
            agg = (
                adf.filter(pl.col("act_kind") != "other")
                .group_by("layer")
                .agg(
                    pl.col("inactivity_fraction").mean().alias("mean_inactive"),
                    pl.col("act_kind").first().alias("kind"),
                )
                .sort("mean_inactive", descending=True)
            )
            if agg.height:
                worst = agg.row(0, named=True)
                thr = self.dead_neuron_threshold
                if worst["mean_inactive"] > thr:
                    kind = worst["kind"]
                    if kind == "relu":
                        label = "dead neurons"
                        fix = "Switch to GELU/LeakyReLU or re-initialise with Kaiming."
                    elif kind == "tanh":
                        label = "saturated (|x|>0.99)"
                        fix = "Reduce LR, add LayerNorm before, or switch to GELU."
                    elif kind == "sigmoid":
                        label = "saturated (|x|>0.99 or |x|<0.01)"
                        fix = "Reduce LR, add BN/LN, or switch to softmax+CE if classification."
                    else:
                        label = "inactive"
                        fix = "Investigate activation distribution."
                    findings["dead_neurons"] = {
                        "severity": "WARNING",
                        "message": (
                            f"'{worst['layer']}' ({kind}): "
                            f"{worst['mean_inactive']:.0%} {label}. {fix}"
                        ),
                    }
                else:
                    findings["dead_neurons"] = {
                        "severity": "HEALTHY",
                        "message": (
                            f"All {agg.height} activation layers healthy "
                            f"(worst: {worst['layer']} at "
                            f"{worst['mean_inactive']:.0%} inactive, below "
                            f"{thr:.0%} threshold)."
                        ),
                    }
            else:
                findings["dead_neurons"] = {
                    "severity": "UNKNOWN",
                    "message": "No activation layers tracked — call track_activations().",
                }
        else:
            findings["dead_neurons"] = {
                "severity": "UNKNOWN",
                "message": "No activation tracking enabled — call track_activations().",
            }

        # 3. Loss trend
        edf = self.epochs_df()
        if edf.height >= 3 and edf["val_loss"].is_not_null().any():
            overfit = self._detect_overfit_epoch()
            train_trend = self._slope(edf["train_loss"].to_list())
            if overfit is not None:
                severity = "WARNING"
                message = (
                    f"Overfitting suspected at epoch {overfit} "
                    "(val loss rising while train loss falls). "
                    "Consider dropout, weight decay, or early stopping."
                )
            elif train_trend > -1e-4 and edf.height >= 5:
                severity = "WARNING"
                message = (
                    f"Underfitting — train loss slope {train_trend:.2e}/epoch. "
                    "Consider a larger model, more epochs, or higher LR."
                )
            else:
                severity = "HEALTHY"
                message = f"Loss converging (train slope {train_trend:.2e}/epoch)."
            findings["loss_trend"] = {"severity": severity, "message": message}
        else:
            findings["loss_trend"] = {
                "severity": "UNKNOWN",
                "message": "Need at least 3 epochs with val_loss for trend analysis.",
            }

        # Human-readable summary
        print("\n" + "═" * 64)
        print("  DL Diagnostics Report — Prescription Pad")
        print("═" * 64)
        icons = {
            "HEALTHY": "✓",
            "WARNING": "!",
            "CRITICAL": "X",
            "UNKNOWN": "?",
        }
        titles = {
            "gradient_flow": "Gradient flow",
            "dead_neurons": "Dead neurons ",
            "loss_trend": "Loss trend   ",
        }
        for key, label in titles.items():
            f = findings[key]
            print(
                f"  [{icons[f['severity']]}] {label} ({f['severity']}): {f['message']}"
            )
        print("═" * 64 + "\n")
        return findings

    # ── Grad-CAM ──────────────────────────────────────────────────────────

    def grad_cam(
        self,
        input_tensor: torch.Tensor,
        target_class: int,
        layer_name: str,
    ) -> torch.Tensor:
        """Compute a Grad-CAM heatmap for ``target_class`` from ``layer_name``.

        Args:
            input_tensor: Input batch ``(N, C, H, W)`` or ``(N, C, L)``.
            target_class: Target class index to explain.
            layer_name: ``model.named_modules()`` key of the conv layer to
                attribute. Usually the last convolutional layer.

        Returns:
            Heatmap tensor with shape matching the spatial dims of the tracked
            layer's output (``(N, H', W')`` for 2D inputs).

        Raises:
            ValueError: If ``layer_name`` is not found in the model.
        """
        target_module: nn.Module | None = None
        for name, module in self.model.named_modules():
            if name == layer_name:
                target_module = module
                break
        if target_module is None:
            raise ValueError(
                f"Layer '{layer_name}' not found in model. "
                f"Available: {[n for n, _ in self.model.named_modules() if n][:10]}..."
            )

        self._gradcam_activation = None
        self._gradcam_gradient = None

        def fwd_hook(_mod: nn.Module, _inp: Any, out: torch.Tensor) -> None:
            self._gradcam_activation = out.detach()

        def bwd_hook(_mod: nn.Module, _gi: Any, go: Any) -> None:
            # go is a tuple — first element is d(output)/d(module-output)
            self._gradcam_gradient = go[0].detach()

        h1 = target_module.register_forward_hook(fwd_hook)
        h2 = target_module.register_full_backward_hook(bwd_hook)
        self._handles.grad_cam.extend([h1, h2])

        # Preserve caller's train/eval state — critical for mid-training use.
        was_training = self.model.training

        try:
            self.model.eval()
            inp = input_tensor.to(self.device)
            logits = self.model(inp)
            if logits.ndim != 2:
                raise ValueError(
                    f"grad_cam expects classification logits (N, C); got {logits.shape}"
                )
            self.model.zero_grad(set_to_none=True)
            one_hot = torch.zeros_like(logits)
            one_hot[:, target_class] = 1.0
            logits.backward(gradient=one_hot, retain_graph=False)

            if self._gradcam_activation is None or self._gradcam_gradient is None:
                raise RuntimeError(
                    "Grad-CAM hooks did not fire — layer may be unreachable "
                    "from the forward path."
                )
            activations = self._gradcam_activation  # (N, K, ...)
            gradients = self._gradcam_gradient  # (N, K, ...)
            # Global-average-pool gradients over spatial dims to get per-channel weights.
            spatial_dims = tuple(range(2, gradients.ndim))
            weights = gradients.mean(dim=spatial_dims, keepdim=True)  # (N, K, 1, ...)
            cam = (weights * activations).sum(dim=1)  # (N, ...)
            cam = torch.relu(cam)
            # Normalise per-sample to [0, 1].
            flat = cam.flatten(start_dim=1)
            mins = flat.min(dim=1, keepdim=True).values
            maxs = flat.max(dim=1, keepdim=True).values
            denom = (maxs - mins).clamp(min=1e-8)
            flat = (flat - mins) / denom
            cam = flat.view_as(cam)
            return cam.cpu()
        finally:
            # Restore caller's train/eval state BEFORE removing hooks, so
            # downstream errors in hook cleanup don't leave model in eval mode.
            if was_training:
                self.model.train()
            h1.remove()
            h2.remove()
            # Remove from bookkeeping too so detach() stays idempotent.
            self._handles.grad_cam = [
                h for h in self._handles.grad_cam if h is not h1 and h is not h2
            ]
            self._gradcam_activation = None
            self._gradcam_gradient = None

    # ── LR range test (static) ────────────────────────────────────────────

    @staticmethod
    def lr_range_test(
        model: nn.Module,
        dataloader: DataLoader,
        *,
        loss_fn: nn.Module | None = None,
        optimizer_cls: type[torch.optim.Optimizer] = torch.optim.SGD,
        lr_min: float = 1e-7,
        lr_max: float = 10.0,
        steps: int = 200,
        device: torch.device | None = None,
        batch_adapter: Callable[[Any], tuple[torch.Tensor, torch.Tensor]] | None = None,
        safety_divisor: float = 10.0,
    ) -> dict[str, Any]:
        """Leslie Smith LR range test (with fastai-style safe-LR recipe).

        Trains for ``steps`` batches while exponentially increasing LR from
        ``lr_min`` to ``lr_max``. Smooths the loss curve with EMA (beta=0.98)
        before finding the steepest-descent point — this matches fastai's
        ``lr_find()`` and avoids picking a single lucky batch in the tail.

        **Critical**: returns BOTH ``min_loss_lr`` (steepest descent) AND
        ``safe_lr = min_loss_lr / safety_divisor`` (default 10). Use ``safe_lr``
        in your optimizer — ``min_loss_lr`` is the edge of instability.

        The model's weights are saved before the test and **restored** on exit
        (via state_dict deepcopy), so calling this does not corrupt your model.

        Args:
            model: The model to probe. Weights are restored after return.
            dataloader: Any DataLoader. By default yields ``(inputs, targets)``
                tuples; pass ``batch_adapter`` for HF/SSL batch formats.
            loss_fn: Loss function (REQUIRED — no silent default).
            optimizer_cls: Optimizer class (default SGD).
            lr_min, lr_max, steps: Sweep configuration.
            device: Override compute device (default: model's current device).
            batch_adapter: Callable ``batch -> (x, y)``. Default handles
                tuple/list; pass a lambda for dict-shaped batches (e.g. HF).
            safety_divisor: Divide steepest-descent LR by this to get safe LR
                (fastai default: 10).

        Returns:
            ``{"safe_lr": float,              # use this in your optimizer
                "min_loss_lr": float,          # steepest descent (edge of instability)
                "divergence_lr": float,        # where smoothed loss > 4× min
                "lrs": [...], "losses": [...], "losses_smooth": [...],
                "figure": go.Figure}``
        """
        if steps < 2:
            raise ValueError("steps must be >= 2")
        if not 0 < lr_min < lr_max:
            raise ValueError("require 0 < lr_min < lr_max")
        if loss_fn is None:
            raise ValueError(
                "loss_fn is required — no silent default. "
                "Pass nn.CrossEntropyLoss() for classification or nn.MSELoss() for regression."
            )

        import copy as _copy

        # Device: honor model's current device unless overridden.
        dev = device or next(model.parameters()).device
        if device is not None:
            model = model.to(dev)

        # Save weights for restoration.
        saved_state = _copy.deepcopy(model.state_dict())

        # Default batch adapter handles tuple/list; raises loudly on dicts.
        def _default_adapter(batch: Any) -> tuple[torch.Tensor, torch.Tensor]:
            if isinstance(batch, (tuple, list)) and len(batch) >= 2:
                return batch[0], batch[1]
            raise ValueError(
                "lr_range_test got a non-(x,y) batch. Pass batch_adapter=... "
                "for HuggingFace-style dict batches or SSL single-tensor batches."
            )

        adapter = batch_adapter or _default_adapter

        try:
            optimizer = optimizer_cls(model.parameters(), lr=lr_min)
            gamma = (lr_max / lr_min) ** (1.0 / (steps - 1))
            lrs: list[float] = []
            losses: list[float] = []
            running_min = float("inf")  # O(1) running min, not O(n)
            model.train()
            data_iter = iter(dataloader)
            for step in range(steps):
                try:
                    batch = next(data_iter)
                except StopIteration:
                    data_iter = iter(dataloader)
                    batch = next(data_iter)
                x, y = adapter(batch)
                x, y = x.to(dev), y.to(dev)
                for pg in optimizer.param_groups:
                    pg["lr"] = lr_min * (gamma**step)
                optimizer.zero_grad(set_to_none=True)
                logits = model(x)
                loss = loss_fn(logits, y)
                loss.backward()
                optimizer.step()
                cur_lr = optimizer.param_groups[0]["lr"]
                cur_loss = float(loss.item())
                lrs.append(cur_lr)
                losses.append(cur_loss)
                if cur_loss < running_min:
                    running_min = cur_loss
                if not math.isfinite(cur_loss) or cur_loss > 10 * running_min:
                    logger.info(
                        "dldiagnostics.lr_range_test.diverged",
                        extra={"step": step, "lr": cur_lr, "loss": cur_loss},
                    )
                    break
        finally:
            # Always restore weights — no silent corruption.
            model.load_state_dict(saved_state)

        # EMA-smoothed losses (fastai convention, beta=0.98).
        beta = 0.98
        losses_smooth: list[float] = []
        ema = 0.0
        for i, ell in enumerate(losses):
            ema = beta * ema + (1 - beta) * ell
            # Bias-correct the EMA.
            losses_smooth.append(ema / (1 - beta ** (i + 1)))

        # min_loss_lr = LR at the steepest-descent point of SMOOTHED loss.
        min_loss_lr = lr_min
        divergence_lr = lr_max
        if len(losses_smooth) >= 3:
            dl = np.diff(np.array(losses_smooth))
            idx = int(np.argmin(dl))
            min_loss_lr = lrs[idx]
            # Divergence = first point where smoothed loss > 4× its minimum.
            min_smooth = min(losses_smooth)
            for i, s in enumerate(losses_smooth):
                if s > 4 * min_smooth and i > idx:
                    divergence_lr = lrs[i]
                    break

        safe_lr = min_loss_lr / safety_divisor

        fig = go.Figure()
        fig.add_trace(
            go.Scatter(
                x=lrs,
                y=losses,
                mode="lines",
                name="raw loss",
                line=dict(color="lightgray"),
            )
        )
        fig.add_trace(
            go.Scatter(
                x=lrs,
                y=losses_smooth,
                mode="lines",
                name="smoothed",
                line=dict(color="#0D9488", width=2),
            )
        )
        fig.add_vline(
            x=safe_lr,
            line=dict(color="#22C55E", dash="dash"),
            annotation_text=f"safe_lr = {safe_lr:.2e}",
        )
        fig.add_vline(
            x=min_loss_lr,
            line=dict(color="#F59E0B", dash="dot"),
            annotation_text=f"min_loss_lr = {min_loss_lr:.2e}",
        )
        fig.update_layout(
            title="LR Range Test (smoothed) — use safe_lr, not min_loss_lr",
            xaxis_title="learning rate",
            yaxis_title="loss",
            xaxis_type="log",
            template="plotly_white",
        )
        logger.info(
            "dldiagnostics.lr_range_test.ok",
            extra={
                "safe_lr": safe_lr,
                "min_loss_lr": min_loss_lr,
                "divergence_lr": divergence_lr,
                "steps_run": len(lrs),
            },
        )
        return {
            "safe_lr": safe_lr,
            "min_loss_lr": min_loss_lr,
            "divergence_lr": divergence_lr,
            "suggested_lr": safe_lr,  # backwards-compat alias
            "lrs": lrs,
            "losses": losses,
            "losses_smooth": losses_smooth,
            "figure": fig,
        }

    # ── Hook factories (internal) ─────────────────────────────────────────

    def _make_grad_hook(self, name: str):
        """Scale-invariant gradient tracking.

        Records three metrics per layer per batch:
          - grad_norm: raw L2 norm (preserved for backwards compatibility)
          - grad_rms: per-element RMS = ‖∇‖ / sqrt(numel) — scale-invariant,
            comparable across layers of any size. This is what LLM training
            dashboards (W&B, TensorBoard) actually display.
          - update_ratio: ‖∇W‖ / ‖W‖ — the "effective LR" per layer.

        Casts to fp32 before reduction so BF16/FP16 gradients don't silently
        produce inf/NaN.
        """
        # Look up the parameter tensor for update-ratio computation.
        try:
            param = dict(self.model.named_parameters())[name]
        except KeyError:
            param = None

        def _hook(grad: torch.Tensor) -> None:
            try:
                # Handle sparse gradients (e.g. nn.Embedding(sparse=True)).
                g = grad.coalesce().values() if grad.is_sparse else grad
                # Cast to fp32 to avoid BF16/FP16 inf.
                g_f32 = g.detach().float()
                l2 = float(g_f32.norm(p=2).item())
                numel = max(g_f32.numel(), 1)
                rms = l2 / (numel**0.5)
                # Update ratio — use detached param weight norm.
                if param is not None:
                    p_norm = float(param.detach().float().norm(p=2).item())
                    upd_ratio = l2 / max(p_norm, 1e-12)
                else:
                    upd_ratio = 0.0
            except Exception as exc:  # pragma: no cover - defensive
                logger.warning(
                    "dldiagnostics.grad_hook.error",
                    extra={"layer": name, "error": str(exc)},
                )
                return
            self._grad_log.append(
                {
                    "batch": self._batch_idx,
                    "layer": name,
                    "grad_norm": l2,  # preserved for compatibility
                    "grad_rms": rms,  # scale-invariant
                    "update_ratio": upd_ratio,  # ‖∇W‖ / ‖W‖
                }
            )

        return _hook

    def _make_act_hook(self, name: str):
        """Activation statistics. Casts to fp32 to avoid BF16/FP16 overflow.

        The `inactivity_fraction` field measures activation-type-appropriate
        pathology:
          - ReLU / GELU / SiLU: fraction with |x| < 1e-6 (dead neurons)
          - Tanh: fraction with |x| > 0.99 (saturated)
          - Sigmoid: fraction with |x| > 0.99 OR |x| < 0.01 (saturated)
          - Others: 0 (no pathology definition)

        The legacy `dead_fraction` field (exact-zero count) is preserved for
        backwards compatibility but is only meaningful for ReLU.
        """
        # Determine activation type from module class name for dispatching.
        act_kind = self._classify_activation_module(name)

        def _hook(_module: nn.Module, _inp: Any, output: torch.Tensor) -> None:
            if not isinstance(output, torch.Tensor) or output.numel() == 0:
                return
            # Cast to fp32 for numerical safety (BF16/FP16 overflow on .std()).
            out = output.detach().float()
            try:
                mean = float(out.mean().item())
                std = float(out.std().item()) if out.numel() > 1 else 0.0
                mn = float(out.min().item())
                mx = float(out.max().item())
                legacy_dead = float((out == 0).float().mean().item())
                # Activation-aware inactivity/saturation metric.
                if act_kind == "relu":
                    inactivity = float((out.abs() < 1e-6).float().mean().item())
                elif act_kind == "tanh":
                    inactivity = float((out.abs() > 0.99).float().mean().item())
                elif act_kind == "sigmoid":
                    inactivity = float(
                        ((out > 0.99) | (out < 0.01)).float().mean().item()
                    )
                else:
                    inactivity = 0.0
            except RuntimeError:
                # Non-numeric tensor (e.g. mixed dtype). Skip silently.
                return
            # Guard against NaN/inf leaking into logs.
            for val_name, val in (("mean", mean), ("std", std)):
                if val != val or val in (float("inf"), float("-inf")):
                    logger.warning(
                        "dldiagnostics.act_hook.nonfinite",
                        extra={"layer": name, "field": val_name},
                    )
                    return
            self._act_log.append(
                {
                    "batch": self._batch_idx,
                    "layer": name,
                    "act_kind": act_kind,
                    "mean": mean,
                    "std": std,
                    "min": mn,
                    "max": mx,
                    "dead_fraction": legacy_dead,
                    "inactivity_fraction": inactivity,
                }
            )

        return _hook

    def _classify_activation_module(self, name: str) -> str:
        """Return one of 'relu', 'tanh', 'sigmoid', 'other' for a module name."""
        try:
            mod = dict(self.model.named_modules())[name]
        except KeyError:
            return "other"
        cls = type(mod).__name__.lower()
        if any(k in cls for k in ("relu", "gelu", "silu", "swish", "elu")):
            return "relu"
        if "tanh" in cls:
            return "tanh"
        if "sigmoid" in cls:
            return "sigmoid"
        return "other"

    def _make_dead_hook(self, name: str):
        def _hook(_module: nn.Module, _inp: Any, output: torch.Tensor) -> None:
            if not isinstance(output, torch.Tensor) or output.numel() == 0:
                return
            out = output.detach()
            # Collapse all non-channel dims. Convention: channel dim = 1 for
            # Conv outputs (N, C, ...); for Linear, output is (N, F) — here
            # we treat dim 1 as "neurons" which matches both shapes.
            if out.ndim < 2:
                return
            reduce_dims = tuple(d for d in range(out.ndim) if d != 1)
            fired = (out != 0).any(dim=reduce_dims).float().cpu()
            if name not in self._firing_counts:
                self._firing_counts[name] = torch.zeros_like(fired)
                self._firing_samples[name] = 0
            self._firing_counts[name] += fired
            self._firing_samples[name] += 1
            # Keep memory bounded by decaying old counts when window exceeded.
            if self._firing_samples[name] > self.window:
                scale = self.window / self._firing_samples[name]
                self._firing_counts[name] = self._firing_counts[name] * scale
                self._firing_samples[name] = self.window

        return _hook

    # ── Internal analysis helpers ─────────────────────────────────────────

    @staticmethod
    def _slope(series: list[float]) -> float:
        """Least-squares slope of a 1D series vs its index (ignores NaN)."""
        xs = np.arange(len(series), dtype=float)
        ys = np.asarray(series, dtype=float)
        mask = np.isfinite(ys)
        if mask.sum() < 2:
            return 0.0
        xs, ys = xs[mask], ys[mask]
        return float(np.polyfit(xs, ys, 1)[0])

    def _detect_overfit_epoch(self) -> int | None:
        """Return epoch index where val loss starts rising while train falls."""
        edf = self.epochs_df()
        if edf.height < 3:
            return None
        train = edf["train_loss"].to_list()
        val = edf["val_loss"].to_list()
        for i in range(2, len(val)):
            v_now, v_prev = val[i], val[i - 1]
            t_now, t_prev = train[i], train[i - 1]
            if any(
                x is None or not math.isfinite(x)
                for x in (v_now, v_prev, t_now, t_prev)
            ):
                continue
            if v_now > v_prev and t_now < t_prev:
                # Confirm trend persists one step back if available.
                return i
        return None


# ════════════════════════════════════════════════════════════════════════
# Standalone diagnostic checkpoint — for use AFTER a training loop
# ════════════════════════════════════════════════════════════════════════


def run_diagnostic_checkpoint(
    model: nn.Module,
    dataloader: DataLoader,
    loss_fn: Callable[..., Any],
    *,
    title: str = "Model",
    n_batches: int = 8,
    train_losses: list[float] | None = None,
    val_losses: list[float] | None = None,
    show: bool = True,
    batch_adapter: Callable[[Any], tuple[torch.Tensor, ...]] | None = None,
) -> tuple[DLDiagnostics, dict[str, Any]]:
    """Run a short instrumented diagnostic pass on a TRAINED model.

    Use this between the Train and Visualise phases of an exercise. It
    attaches the four diagnostic instruments (gradients, activations,
    dead-neurons, scalar history) to the model, runs ``n_batches`` of
    forward-backward passes to populate the history, replays any
    epoch-level losses you collected during real training, and prints the
    Prescription Pad with auto-diagnosis.

    The model's weights are NOT updated — gradients are computed but no
    optimizer step is taken. The model's train/eval state is preserved.

    Args:
        model: A trained ``nn.Module``.
        dataloader: Any DataLoader whose batches the loss function accepts.
        loss_fn: Callable. The default contract is
            ``loss_fn(model, batch) -> (scalar_loss, extras_dict)`` matching
            the M5 exercise convention. Pass ``batch_adapter`` to wrap a
            different signature.
        title: Human-readable name printed in the dashboard title.
        n_batches: How many batches to instrument (default 8 — enough for
            a stable mean per layer without slowing the exercise down).
        train_losses: Optional list of per-epoch train losses captured
            during the actual training run. Replayed into the dashboard's
            stethoscope view so students see the real loss trajectory, not
            just the diagnostic-pass losses.
        val_losses: Optional list of per-epoch validation losses, same
            length as ``train_losses``.
        show: If ``True``, calls ``.show()`` on the dashboard figure.
        batch_adapter: Optional ``batch -> (loss_fn_args...)`` translator
            for non-standard batch shapes.

    Returns:
        ``(diag, findings)`` so the caller can inspect the DataFrames and
        the Prescription Pad findings dict.
    """
    if not isinstance(model, nn.Module):
        raise TypeError("run_diagnostic_checkpoint requires an nn.Module")
    if n_batches < 1:
        raise ValueError("n_batches must be >= 1")

    diag = DLDiagnostics(model)
    diag.track_gradients().track_activations().track_dead_neurons()

    # Replay real training history into the dashboard if provided.
    if train_losses or val_losses:
        n_epochs = max(len(train_losses or []), len(val_losses or []))
        for i in range(n_epochs):
            tl = (
                float(train_losses[i])
                if train_losses and i < len(train_losses)
                else None
            )
            vl = float(val_losses[i]) if val_losses and i < len(val_losses) else None
            # Synthesise one batch entry per epoch so the per-batch stethoscope
            # has data to plot — students still see the real epoch losses.
            if tl is not None:
                diag.record_batch(loss=tl)
            diag.record_epoch(train_loss=tl, val_loss=vl)

    was_training = model.training
    try:
        model.train()  # Enable gradients + activation hooks.
        seen = 0
        for batch in dataloader:
            if seen >= n_batches:
                break
            try:
                if batch_adapter is not None:
                    args = batch_adapter(batch)
                    loss_out = loss_fn(model, *args)
                else:
                    loss_out = loss_fn(model, batch)
                # Convention: M5 loss_fn returns (loss, extras_dict). Allow
                # either a bare tensor or a tuple.
                if isinstance(loss_out, tuple):
                    loss = loss_out[0]
                else:
                    loss = loss_out
                if not isinstance(loss, torch.Tensor):
                    raise TypeError(
                        f"loss_fn returned {type(loss).__name__}; expected Tensor"
                    )
                model.zero_grad(set_to_none=True)
                loss.backward()
                # NOTE: no optimizer.step() — diagnostic pass is read-only.
                diag.record_batch(loss=float(loss.item()))
            except Exception as exc:  # pragma: no cover - student loop variability
                logger.warning(
                    "dldiagnostics.checkpoint.batch_skipped",
                    extra={"batch_idx": seen, "error": str(exc)},
                )
            seen += 1
    finally:
        if not was_training:
            model.eval()

    # Render the dashboard and the Prescription Pad.
    fig = diag.plot_training_dashboard()
    fig.update_layout(title=f"{title} — Diagnostic Dashboard (Vital Signs)")
    if show:
        try:
            fig.show()
        except Exception as exc:  # pragma: no cover - no display in CI
            logger.info(
                "dldiagnostics.checkpoint.show_skipped",
                extra={"error": str(exc)},
            )

    findings = diag.report()
    return diag, findings


def diagnose_classifier(
    model: nn.Module,
    dataloader: DataLoader,
    *,
    title: str = "Classifier",
    n_batches: int = 8,
    train_losses: list[float] | None = None,
    val_losses: list[float] | None = None,
    show: bool = True,
    forward_returns_tuple: bool = False,
) -> tuple[DLDiagnostics, dict[str, Any]]:
    """Convenience wrapper for ``(x, y)`` cross-entropy classifiers.

    Equivalent to :func:`run_diagnostic_checkpoint` with a built-in
    ``loss_fn`` that calls ``F.cross_entropy(model(x), y)``. Use this for
    CNN, transformer, and transfer-learning exercises.

    Args:
        model: Trained classifier ``nn.Module``.
        dataloader: Yields ``(x, y)`` tuples; ``y`` is class indices.
        title: Title for the dashboard.
        n_batches: Batches to instrument.
        train_losses, val_losses: Optional epoch histories to replay.
        show: Show the figure inline.
        forward_returns_tuple: If ``True``, ``model(x)`` returns
            ``(logits, ...)`` (e.g. attention models) — only the first
            element is used as logits.

    Returns:
        ``(diag, findings)``
    """
    import torch.nn.functional as F  # local — torch already imported

    def _cls_loss(m: nn.Module, batch: Any) -> torch.Tensor:
        x, y = batch[0], batch[1]
        out = m(x)
        logits = out[0] if forward_returns_tuple else out
        return F.cross_entropy(logits, y)

    return run_diagnostic_checkpoint(
        model,
        dataloader,
        _cls_loss,
        title=title,
        n_batches=n_batches,
        train_losses=train_losses,
        val_losses=val_losses,
        show=show,
    )


def diagnose_regressor(
    model: nn.Module,
    dataloader: DataLoader,
    *,
    title: str = "Regressor",
    n_batches: int = 8,
    train_losses: list[float] | None = None,
    val_losses: list[float] | None = None,
    show: bool = True,
    forward_returns_tuple: bool = False,
) -> tuple[DLDiagnostics, dict[str, Any]]:
    """Convenience wrapper for ``(x, y)`` MSE regressors (RNN exercises).

    Equivalent to :func:`run_diagnostic_checkpoint` with a built-in
    ``loss_fn`` that calls ``F.mse_loss(model(x), y)``.

    Args:
        forward_returns_tuple: Set ``True`` for attention models that
            return ``(predictions, attn_weights)``.
    """
    import torch.nn.functional as F

    def _reg_loss(m: nn.Module, batch: Any) -> torch.Tensor:
        x, y = batch[0], batch[1]
        out = m(x)
        pred = out[0] if forward_returns_tuple else out
        return F.mse_loss(pred, y)

    return run_diagnostic_checkpoint(
        model,
        dataloader,
        _reg_loss,
        title=title,
        n_batches=n_batches,
        train_losses=train_losses,
        val_losses=val_losses,
        show=show,
    )


__all__ = [
    "DLDiagnostics",
    "run_diagnostic_checkpoint",
    "diagnose_classifier",
    "diagnose_regressor",
]

# ── ex_3.py ──
"""
# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 3: Shared Utilities
# ════════════════════════════════════════════════════════════════════════
#
# Common data loading, windowing, training, visualisation, and experiment
# tracking utilities shared across all technique files in this exercise.
#
# This module is NOT meant to be run standalone — import it from the
# technique files (01_vanilla_rnn.py, 02_lstm.py, etc.).
# ════════════════════════════════════════════════════════════════════════
"""

import asyncio
import pickle
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


from kailash.db import ConnectionManager
from kailash_ml import ModelVisualizer
from kailash_ml.engines.experiment_tracker import ExperimentTracker
from kailash_ml.engines.model_registry import ModelRegistry

# ── Constants ───────────────────────────────────────────────────────────
REPO_ROOT = Path(__file__).resolve().parents[2]
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "stocks"
OUTPUT_DIR = REPO_ROOT / "outputs" / "mlfp05" / "ex_3"

TICKERS = {
    "^STI": "Straits Times Index",
    "DBS.SI": "DBS Group",
    "9988.HK": "Alibaba HK",
    "AAPL": "Apple",
    "005930.KS": "Samsung",
    "7203.T": "Toyota",
}

SEQ_LEN = 20  # 20-day lookback (4 trading weeks)
FORECAST_HORIZON = 5  # predict next 5 days
FEATURES = ["Close", "High", "Low", "Volume"]
HIDDEN_DIM = 64
EPOCHS = 15
LR = 1e-3
CLIP = 1.0
BATCH_SIZE = 64

def init_environment() -> torch.device:
    """Set up environment, seeds, device, and output directories."""
    setup_environment()
    torch.manual_seed(42)
    np.random.seed(42)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    device = get_device()
    print(f"Using device: {device}")
    return device

# ── Data Loading ────────────────────────────────────────────────────────
def fetch_ticker(symbol: str) -> pl.DataFrame:
    """Download daily OHLCV bars from yfinance, return polars DataFrame."""
    import yfinance as yf

    df = yf.download(
        symbol, start="2010-01-01", end="2024-12-31", progress=False, auto_adjust=True
    )
    if df is None or len(df) == 0:
        raise RuntimeError(f"yfinance returned empty frame for {symbol}")
    df = df.copy()
    df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
    return pl.from_pandas(df.reset_index())

def load_or_fetch(symbol: str) -> tuple[pl.DataFrame | None, str]:
    """Load from parquet cache, or download and cache."""
    cache = DATA_DIR / f"{symbol.replace('^', '').replace('.', '_')}.parquet"
    if cache.exists():
        return pl.read_parquet(cache), "cache"
    try:
        df = fetch_ticker(symbol)
        df.write_parquet(cache)
        return df, "yfinance"
    except Exception as exc:
        print(f"  {symbol} unavailable ({type(exc).__name__}: {exc})")
        return None, "failed"

def load_stock_data() -> tuple[dict[str, pl.DataFrame], str, pl.DataFrame]:
    """Load all tickers and return (stock_data, primary_symbol, primary_df)."""
    stock_data: dict[str, pl.DataFrame] = {}
    for symbol, name in TICKERS.items():
        df, source = load_or_fetch(symbol)
        if df is not None:
            stock_data[symbol] = df
            print(f"  {symbol} ({name}): {len(df)} days [{source}]")

    if "^STI" not in stock_data and "AAPL" not in stock_data:
        raise RuntimeError("Need at least ^STI or AAPL data to proceed")

    primary = "^STI" if "^STI" in stock_data else "AAPL"
    primary_df = stock_data[primary]
    print(
        f"\nPrimary: {primary} -- {len(primary_df)} days, "
        f"{primary_df['Date'].min()} -> {primary_df['Date'].max()}"
    )
    return stock_data, primary, primary_df

# ── Windowed Datasets ───────────────────────────────────────────────────
def build_dataset(
    df: pl.DataFrame,
    seq_len: int = SEQ_LEN,
    horizon: int = FORECAST_HORIZON,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, int]:
    """Build (seq_len window) -> (next horizon closes) arrays with z-score normalisation.

    Returns: X, y, mean, std, n_train_windows
    """
    data = df.select(FEATURES).to_numpy().astype(np.float32)
    n = len(data)
    split_n = int(0.8 * n)
    train_data = data[:split_n]
    mean = train_data.mean(axis=0, keepdims=True)
    std = train_data.std(axis=0, keepdims=True) + 1e-8
    data_norm = (data - mean) / std

    n_windows = n - seq_len - horizon + 1
    X = np.stack([data_norm[i : i + seq_len] for i in range(n_windows)])
    y = np.stack(
        [data_norm[i + seq_len : i + seq_len + horizon, 0] for i in range(n_windows)]
    )
    split_idx = split_n - seq_len
    return X.astype(np.float32), y.astype(np.float32), mean, std, split_idx

def prepare_dataloaders(
    primary_df: pl.DataFrame,
    device: torch.device,
) -> tuple[
    DataLoader,
    DataLoader,
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    np.ndarray,
    np.ndarray,
    int,
    int,
]:
    """Build train/val dataloaders and return raw tensors plus normalisation stats.

    Returns: train_loader, val_loader, X_train_t, y_train_t, X_val_t, y_val_t,
             norm_mean, norm_std, n_train_w, n_features
    """
    X_all, y_all, norm_mean, norm_std, n_train_w = build_dataset(primary_df)
    print(
        f"Built {len(X_all)} windows (seq_len={SEQ_LEN}, horizon={FORECAST_HORIZON}); "
        f"train {n_train_w}, val {len(X_all) - n_train_w}"
    )

    X_train_t = torch.from_numpy(X_all[:n_train_w]).to(device)
    y_train_t = torch.from_numpy(y_all[:n_train_w]).to(device)
    X_val_t = torch.from_numpy(X_all[n_train_w:]).to(device)
    y_val_t = torch.from_numpy(y_all[n_train_w:]).to(device)
    print(f"  X_train: {tuple(X_train_t.shape)}  y_train: {tuple(y_train_t.shape)}")
    print(f"  X_val:   {tuple(X_val_t.shape)}    y_val:   {tuple(y_val_t.shape)}")

    train_loader = DataLoader(
        TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True
    )
    val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE)
    n_features = X_train_t.shape[-1]

    return (
        train_loader,
        val_loader,
        X_train_t,
        y_train_t,
        X_val_t,
        y_val_t,
        norm_mean,
        norm_std,
        n_train_w,
        n_features,
    )

# ── Experiment Tracking ─────────────────────────────────────────────────
async def _setup_engines(
    primary: str,
    experiment_suffix: str = "",
) -> tuple[ConnectionManager, ExperimentTracker, str, ModelRegistry | None, bool]:
    """Create ExperimentTracker and ModelRegistry for a technique file."""
    conn = ConnectionManager("sqlite:///mlfp05_rnns.db")
    await conn.initialize()

    tracker = ExperimentTracker(conn)
    exp_name = await tracker.create_experiment(
        name=(
            f"m5_rnns_{experiment_suffix}"
            if experiment_suffix
            else "m5_rnns_sequence_models"
        ),
        description=(
            f"RNN variant on {primary} stock data. "
            f"Multi-step forecasting (next {FORECAST_HORIZON} days)."
        ),
    )

    try:
        registry = ModelRegistry(conn)
        has_registry = True
    except Exception as e:
        registry = None
        has_registry = False
        print(f"  Note: ModelRegistry setup skipped ({e})")

    return conn, tracker, exp_name, registry, has_registry

def setup_engines(
    primary: str,
    experiment_suffix: str = "",
) -> tuple[ConnectionManager, ExperimentTracker, str, ModelRegistry | None, bool]:
    """Sync wrapper for engine setup."""
    return asyncio.run(_setup_engines(primary, experiment_suffix))

# ── Training Harness ────────────────────────────────────────────────────
def compute_gradient_norm(model: nn.Module) -> float:
    """Compute the total L2 norm of all gradients (before clipping)."""
    total_norm = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total_norm += p.grad.data.norm(2).item() ** 2
    return total_norm**0.5

def _predict(model: nn.Module, x: torch.Tensor, attn: bool = False) -> torch.Tensor:
    """Forward pass, handling attention models that return a tuple."""
    out = model(x)
    return out[0] if attn else out

async def _train_model_async(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    epochs: int = EPOCHS,
    lr: float = LR,
    clip: float = CLIP,
    attn: bool = False,
) -> dict[str, Any]:
    """Train with gradient tracking, log to ExperimentTracker."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    train_losses, val_losses, gradient_norms = [], [], []
    n_params = sum(p.numel() for p in model.parameters())

    async with tracker.run(experiment_name=exp_name, run_name=name) as ctx:
        await ctx.log_params(
            {
                "model_type": name,
                "hidden_dim": str(HIDDEN_DIM),
                "seq_len": str(SEQ_LEN),
                "forecast_horizon": str(FORECAST_HORIZON),
                "epochs": str(epochs),
                "lr": str(lr),
                "clip_norm": str(clip),
                "n_params": str(n_params),
            }
        )
        print(f"  [{name}] {n_params:,} parameters")

        for epoch in range(epochs):
            model.train()
            b_losses, e_grads = [], []
            for xb, yb in train_loader:
                opt.zero_grad()
                loss = F.mse_loss(_predict(model, xb, attn), yb)
                loss.backward()
                e_grads.append(compute_gradient_norm(model))
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip)
                opt.step()
                b_losses.append(loss.item())

            tl, gn = float(np.mean(b_losses)), float(np.mean(e_grads))
            train_losses.append(tl)
            gradient_norms.append(gn)

            model.eval()
            with torch.no_grad():
                vl = float(
                    np.mean(
                        [
                            F.mse_loss(_predict(model, xb, attn), yb).item()
                            for xb, yb in val_loader
                        ]
                    )
                )
            val_losses.append(vl)

            await ctx.log_metrics(
                {"train_loss": tl, "val_loss": vl, "gradient_norm": gn},
                step=epoch + 1,
            )
            print(
                f"  [{name}] epoch {epoch+1:2d}/{epochs}  "
                f"train={tl:.4f}  val={vl:.4f}  grad={gn:.4f}"
            )

        await ctx.log_metric("final_val_loss", val_losses[-1])

    return {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "gradient_norms": gradient_norms,
        "final_val_loss": val_losses[-1],
    }

def train_model(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    epochs: int = EPOCHS,
    lr: float = LR,
    clip: float = CLIP,
    attn: bool = False,
) -> dict[str, Any]:
    """Sync wrapper for training."""
    return asyncio.run(
        _train_model_async(
            model,
            name,
            tracker,
            exp_name,
            train_loader,
            val_loader,
            device,
            epochs,
            lr,
            clip,
            attn,
        )
    )

# ── Model Registry ──────────────────────────────────────────────────────
def register_best_model(
    model: nn.Module,
    model_name: str,
    val_loss: float,
    primary: str,
    registry: ModelRegistry | None,
    has_registry: bool,
) -> None:
    """Register a model in the ModelRegistry."""
    if not has_registry or registry is None:
        print("  ModelRegistry not available, skipping registration")
        return

    model_bytes = pickle.dumps(model.state_dict())
    try:
        reg_result = asyncio.run(
            registry.register(
                name=f"m5_rnn_{model_name.lower()}_{primary.replace('^', '')}",
                model_data=model_bytes,
                metadata={
                    "architecture": model_name,
                    "ticker": primary,
                    "hidden_dim": HIDDEN_DIM,
                    "seq_len": SEQ_LEN,
                    "forecast_horizon": FORECAST_HORIZON,
                    "val_loss": val_loss,
                    "epochs": EPOCHS,
                },
            )
        )
        print(f"  Registered: {reg_result}")
    except Exception as e:
        print(f"  ModelRegistry registration skipped ({type(e).__name__}: {e})")

# ── Visualisation Helpers ───────────────────────────────────────────────
def get_visualizer() -> ModelVisualizer:
    """Create a ModelVisualizer instance."""
    return ModelVisualizer()

def plot_training_curves(
    viz: ModelVisualizer,
    results: dict[str, Any],
    model_name: str,
    output_prefix: str,
) -> None:
    """Plot training/validation loss curves and gradient norms."""
    train_metrics = {
        f"{model_name} train": results["train_losses"],
        f"{model_name} val": results["val_losses"],
    }
    viz.training_history(
        metrics=train_metrics, x_label="Epoch", y_label="MSE Loss"
    ).write_html(str(OUTPUT_DIR / f"{output_prefix}_training_curves.html"))

    grad_metrics = {model_name: results["gradient_norms"]}
    viz.training_history(
        metrics=grad_metrics, x_label="Epoch", y_label="Gradient L2 Norm"
    ).write_html(str(OUTPUT_DIR / f"{output_prefix}_gradient_norms.html"))

def plot_predictions(
    viz: ModelVisualizer,
    model: nn.Module,
    X_val_t: torch.Tensor,
    y_val_t: torch.Tensor,
    norm_mean: np.ndarray,
    norm_std: np.ndarray,
    output_prefix: str,
    attn: bool = False,
) -> tuple[np.ndarray, np.ndarray, torch.Tensor | None]:
    """Generate prediction-vs-actual scatter and return denormalised arrays.

    Returns: preds_denorm, actual_denorm, attn_weights (or None)
    """
    model.eval()
    with torch.no_grad():
        if attn:
            val_preds, attn_weights = model(X_val_t)
        else:
            val_preds = model(X_val_t)
            attn_weights = None

    close_mean, close_std = norm_mean[0, 0], norm_std[0, 0]
    preds_denorm = val_preds.cpu().numpy() * close_std + close_mean
    actual_denorm = y_val_t.cpu().numpy() * close_std + close_mean

    pred_df = pl.DataFrame(
        {
            "actual": actual_denorm[:, 0].tolist(),
            "predicted": preds_denorm[:, 0].tolist(),
        }
    )
    viz.scatter(pred_df, x="actual", y="predicted").write_html(
        str(OUTPUT_DIR / f"{output_prefix}_pred_vs_actual.html")
    )

    return preds_denorm, actual_denorm, attn_weights

def plot_time_series_overlay(
    preds_denorm: np.ndarray,
    actual_denorm: np.ndarray,
    output_prefix: str,
    title: str = "Predicted vs Actual Close Price",
    n_points: int = 200,
) -> None:
    """Plot predicted vs actual as overlaid time-series lines."""
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    n = min(n_points, len(preds_denorm))
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(
        range(n), actual_denorm[:n, 0], label="Actual", color="#2196F3", linewidth=1.5
    )
    ax.plot(
        range(n),
        preds_denorm[:n, 0],
        label="Predicted",
        color="#FF5722",
        linewidth=1.5,
        linestyle="--",
        alpha=0.85,
    )
    ax.set_xlabel("Validation Window Index")
    ax.set_ylabel("Close Price")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(str(OUTPUT_DIR / f"{output_prefix}_time_series_overlay.png"), dpi=150)
    plt.close(fig)
    print(f"  Saved: {output_prefix}_time_series_overlay.png")

def plot_horizon_error(
    preds_denorm: np.ndarray,
    actual_denorm: np.ndarray,
    model_name: str,
) -> list[float]:
    """Print and return RMSE by forecast horizon day."""
    print(f"\n  Forecast Error by Horizon Day ({model_name}):")
    rmses = []
    for day in range(FORECAST_HORIZON):
        rmse = (
            float(np.mean((preds_denorm[:, day] - actual_denorm[:, day]) ** 2)) ** 0.5
        )
        rmses.append(rmse)
        print(f"    Day {day + 1}: RMSE={rmse:.2f}")
    return rmses



# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 3.3: GRU as a Lightweight Alternative
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this section, you will be able to:
#   - Explain the GRU vs LSTM tradeoff (fewer parameters, similar accuracy)
#   - Build a GRU regressor with torch.nn.GRU for multi-step forecasting
#   - Compare parameter counts: GRU uses ~75% of LSTM's parameters
#   - Measure inference latency differences between GRU and LSTM
#   - Visualise hidden state dynamics unique to GRU's architecture
#   - Track training with ExperimentTracker
#
# PREREQUISITES: 02_lstm.py (understand gating mechanisms).
# ESTIMATED TIME: ~25-30 min
#
# DATASET: STI + APAC/global stocks via yfinance (2010-2024).
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import time

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

    BATCH_SIZE,
    CLIP,
    EPOCHS,
    FORECAST_HORIZON,
    HIDDEN_DIM,
    LR,
    OUTPUT_DIR,
    SEQ_LEN,
    init_environment,
    load_stock_data,
    prepare_dataloaders,
    setup_engines,
    train_model,
    register_best_model,
    get_visualizer,
    plot_training_curves,
    plot_predictions,
    plot_time_series_overlay,
    plot_horizon_error,
)



## THEORY — GRU: Simpler Gates, Similar Power


In [ ]:
#
# The GRU (Gated Recurrent Unit) was introduced in 2014 by Cho et al.
# as a simplification of LSTM. The key differences:
#
# LSTM has 3 gates + cell state:
#   forget, input, output gates + separate cell state C_t
#   Parameters: 4 * (input_dim + hidden_dim) * hidden_dim
#
# GRU has 2 gates, NO separate cell state:
#   z_t = sigma(W_z [h_{t-1}, x_t])          UPDATE gate
#       "How much of the old hidden state to keep?"
#       Combines LSTM's forget + input gates into one decision
#
#   r_t = sigma(W_r [h_{t-1}, x_t])          RESET gate
#       "How much of the old hidden state to use when computing the candidate?"
#       When r=0, the candidate ignores history — like starting fresh
#
#   h_tilde = tanh(W [r_t * h_{t-1}, x_t])   CANDIDATE hidden state
#       The reset gate filters what history is relevant for the candidate
#
#   h_t = (1 - z_t) * h_{t-1} + z_t * h_tilde   HIDDEN STATE UPDATE
#       Linear interpolation: z=1 means "fully update", z=0 means "keep old"
#       This is equivalent to LSTM's cell highway, but simpler
#
# INTUITION for non-technical professionals:
#   If LSTM is a notebook with separate pages for long-term and short-term
#   notes, GRU is a single page where you decide how much to erase before
#   writing new notes. Simpler, faster to use, and usually just as effective.
#
# WHEN TO CHOOSE GRU OVER LSTM:
#   - Real-time systems where latency matters (fewer computations per step)
#   - Smaller datasets where fewer parameters reduce overfitting risk
#   - Mobile/edge deployment where model size matters
#   - When empirical testing shows similar accuracy (which is often)
#
# WHEN LSTM IS BETTER:
#   - Very long sequences (>100 steps) where the separate cell state helps
#   - Tasks requiring fine-grained memory control (e.g., language modelling)
#   - When you have abundant data and compute to train the extra parameters



In [ ]:
device = init_environment()



## TASK 1 — Load data and set up experiment tracking


In [ ]:
stock_data, PRIMARY, primary_df = load_stock_data()

(
    train_loader,
    val_loader,
    X_train_t,
    y_train_t,
    X_val_t,
    y_val_t,
    norm_mean,
    norm_std,
    n_train_w,
    N_FEATURES,
) = prepare_dataloaders(primary_df, device)

conn, tracker, exp_name, registry, has_registry = setup_engines(
    PRIMARY, experiment_suffix="gru"
)



### Checkpoint 1


In [ ]:
assert X_train_t.shape[1] == SEQ_LEN
assert tracker is not None
print("--- Checkpoint 1 passed --- data and tracking ready\n")



## TASK 2 — Build the GRU and LSTM for comparison


LSTM for direct comparison with GRU.


In [ ]:
class GRURegressor(nn.Module):
    def __init__(
        self, input_dim: int, hidden_dim: int, horizon: int = FORECAST_HORIZON
    ):
        super().__init__()
        # TODO: Define GRU layer — nn.GRU(input_dim, hidden_dim, batch_first=True)
        # TODO: Define prediction head — nn.Linear(hidden_dim, horizon)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Pass x through self.gru -> out, _
        # TODO: Return self.head(out[:, -1]) — last hidden state -> (batch, horizon)
        pass


class LSTMRegressor(nn.Module):

    def __init__(
        self, input_dim: int, hidden_dim: int, horizon: int = FORECAST_HORIZON
    ):
        super().__init__()
        # TODO: Define LSTM layer — nn.LSTM(input_dim, hidden_dim, batch_first=True)
        # TODO: Define prediction head — nn.Linear(hidden_dim, horizon)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Pass x through self.lstm -> out, _
        # TODO: Return self.head(out[:, -1])
        pass


gru_model = GRURegressor(input_dim=N_FEATURES, hidden_dim=HIDDEN_DIM)
lstm_compare = LSTMRegressor(input_dim=N_FEATURES, hidden_dim=HIDDEN_DIM)

n_params_gru = sum(p.numel() for p in gru_model.parameters())
n_params_lstm = sum(p.numel() for p in lstm_compare.parameters())
param_ratio = n_params_gru / n_params_lstm * 100

print(f"GRU parameters:  {n_params_gru:,}")
print(f"LSTM parameters: {n_params_lstm:,}")
print(
    f"GRU is {param_ratio:.0f}% of LSTM parameter count ({100-param_ratio:.0f}% fewer)"
)



### Checkpoint 2


In [ ]:
assert n_params_gru < n_params_lstm, "GRU should have fewer parameters than LSTM"
assert param_ratio < 85, f"GRU should be ~75% of LSTM params, got {param_ratio:.0f}%"
dummy_input = torch.randn(2, SEQ_LEN, N_FEATURES, device=device)
gru_model.to(device)
dummy_out = gru_model(dummy_input)
assert dummy_out.shape == (2, FORECAST_HORIZON)
print("--- Checkpoint 2 passed --- GRU architecture verified\n")



## TASK 3 — Train the GRU


In [ ]:
print(f"\n== Training GRU on {PRIMARY} ==")
gru_results = train_model(
    gru_model,
    "GRU",
    tracker,
    exp_name,
    train_loader,
    val_loader,
    device,
)



### Checkpoint 3


In [ ]:
assert len(gru_results["train_losses"]) == EPOCHS
assert gru_results["final_val_loss"] < 5.0
print(f"\n  Final val loss: {gru_results['final_val_loss']:.4f}")
print("--- Checkpoint 3 passed --- GRU trained\n")



## TASK 4 — Train LSTM for head-to-head comparison


In [ ]:
print(f"\n== Training LSTM (comparison) on {PRIMARY} ==")
lstm_results = train_model(
    lstm_compare,
    "LSTM_compare",
    tracker,
    exp_name,
    train_loader,
    val_loader,
    device,
)

print(f"\n  == Head-to-Head Comparison ==")
print(f"  {'Metric':<25s} {'GRU':>10s} {'LSTM':>10s}")
print(f"  {'Parameters':.<25s} {n_params_gru:>10,d} {n_params_lstm:>10,d}")
print(
    f"  {'Final val loss':.<25s} {gru_results['final_val_loss']:>10.4f} {lstm_results['final_val_loss']:>10.4f}"
)
print(
    f"  {'Avg grad norm':.<25s} {np.mean(gru_results['gradient_norms']):>10.4f} {np.mean(lstm_results['gradient_norms']):>10.4f}"
)



### Checkpoint 4


In [ ]:
assert lstm_results["final_val_loss"] < 5.0
print("--- Checkpoint 4 passed --- head-to-head comparison complete\n")



## TASK 5 — Visualise: inference latency comparison


Measure average inference latency in milliseconds.


In [ ]:
# In production systems (real-time sensor monitoring, trading), inference
# latency matters as much as accuracy. GRU's fewer operations per step
# translate to measurable speed gains.


def benchmark_inference(model: nn.Module, name: str, n_runs: int = 100) -> float:
    model.eval()
    test_input = torch.randn(1, SEQ_LEN, N_FEATURES, device=device)

    # Warmup
    with torch.no_grad():
        for _ in range(10):
            model(test_input)

    # Timed runs
    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_runs):
            model(test_input)

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_ms = (time.perf_counter() - start) / n_runs * 1000
    print(f"  {name}: {elapsed_ms:.3f} ms/inference (avg over {n_runs} runs)")
    return elapsed_ms


print("\n== Inference Latency Comparison ==")
gru_latency = benchmark_inference(gru_model, "GRU")
lstm_latency = benchmark_inference(lstm_compare, "LSTM")
speedup = lstm_latency / max(gru_latency, 1e-6)
print(f"  GRU speedup: {speedup:.2f}x faster than LSTM")

# TODO: Create side-by-side bar charts (2 subplots, figsize 14x5):
#   Left: Inference latency bars for GRU (green) and LSTM (blue)
#     - Add ms values as text above each bar
#   Right: Parameter count bars for GRU (green) and LSTM (blue)
#     - Add comma-formatted counts above each bar
#   Suptitle: f"GRU vs LSTM: {param_ratio:.0f}% parameters, {speedup:.2f}x faster"
#   Save to OUTPUT_DIR / "03_gru_latency_comparison.png"
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
# TODO: Fill in both bar chart subplots
fig.tight_layout()
fig.savefig(str(OUTPUT_DIR / "03_gru_latency_comparison.png"), dpi=150)
plt.close(fig)
print("  Saved: 03_gru_latency_comparison.png")



### Checkpoint 5


In [ ]:
assert gru_latency > 0 and lstm_latency > 0
assert (OUTPUT_DIR / "03_gru_latency_comparison.png").exists()
print("--- Checkpoint 5 passed --- latency comparison complete\n")



## TASK 6 — Visualise: GRU hidden state dynamics


Extract and visualise GRU gate activations using hooks.


In [ ]:
# GRU's update and reset gates control how information flows.
# Visualise how the hidden state evolves differently from LSTM.


def visualise_gru_gates(model: nn.Module, sample: torch.Tensor) -> None:
    model.eval()

    # TODO: Run step-by-step through the GRU to capture hidden states
    #   gru_layer = model.gru
    #   Initialise h = zeros(1, 1, HIDDEN_DIM) on device
    #   Loop through timesteps: out, h = gru_layer(sample[:, t:t+1, :], h)
    #   Append h.squeeze().cpu().numpy() to hidden_states list
    # TODO: Stack into hidden_matrix of shape (seq_len, hidden_dim)

    # TODO: Create 2-row subplot (14, 8):
    #   Top: heatmap of hidden_matrix.T with "RdBu_r" cmap
    #     Title: "GRU Hidden State Evolution (all dimensions)"
    #   Bottom: line plot of top-5 most active dimensions by variance
    #     Title includes "(sharper transitions = update gate)"
    #   Suptitle: "GRU Hidden State Dynamics"
    #   Save to OUTPUT_DIR / "03_gru_hidden_dynamics.png"
    pass


sample_input = X_val_t[:1]
visualise_gru_gates(gru_model, sample_input)



## TASK 7 — Visualise: predicted vs actual + training curves


In [ ]:
viz = get_visualizer()
plot_training_curves(viz, gru_results, "GRU", "03_gru")

preds_denorm, actual_denorm, _ = plot_predictions(
    viz, gru_model, X_val_t, y_val_t, norm_mean, norm_std, "03_gru"
)

plot_time_series_overlay(
    preds_denorm,
    actual_denorm,
    "03_gru",
    title=f"GRU: Predicted vs Actual Close ({PRIMARY})",
)

rmses = plot_horizon_error(preds_denorm, actual_denorm, "GRU")



### Checkpoint 6


In [ ]:
assert (OUTPUT_DIR / "03_gru_training_curves.html").exists()
assert (OUTPUT_DIR / "03_gru_time_series_overlay.png").exists()
assert (OUTPUT_DIR / "03_gru_hidden_dynamics.png").exists()
print("--- Checkpoint 6 passed --- GRU visualisations generated\n")



## TASK 8 — Register model


In [ ]:
register_best_model(
    gru_model,
    "GRU",
    gru_results["final_val_loss"],
    PRIMARY,
    registry,
    has_registry,
)



## APPLY — SMRT Predictive Maintenance: Real-Time Sensor Monitoring


In [ ]:
#
# BUSINESS SCENARIO:
#   You are a data engineer at SMRT Corporation, which operates
#   Singapore's MRT (Mass Rapid Transit) network carrying ~3.4 million
#   trips per day. Train wheels, bearings, and axles generate vibration
#   data captured by accelerometers at 1-second intervals.
#
# WHY GRU (NOT LSTM)?
#   Predictive maintenance on rail infrastructure requires REAL-TIME
#   inference — the model must predict the next minute's vibration
#   reading before the sensor captures it. With 200+ sensors per train
#   and 200+ trains running simultaneously:
#     - At 200 sensors x 200 trains x 60 readings/min = 2.4M inferences/min
#     - GRU's speed advantage means more sensors served per second
#
# DELIVERABLES:
#   - Next-minute vibration prediction with anomaly threshold
#   - Latency comparison: can GRU serve all sensors in real-time?
#   - Maintenance alert: "bearing X shows increasing vibration trend"
print("\n" + "=" * 70)
print("  APPLY: SMRT Predictive Maintenance — Vibration Monitoring")
print("=" * 70)

# TODO: Generate realistic vibration sensor data
#   - n_readings = 60 * 24 * 30 (30 days at 1-minute intervals)
#   - Base vibration: 2.5 + 0.3 * sin(daily cycle)
#   - Add gradual bearing degradation in the last 5 days (0 -> 1.5 increase)
#   - Add Gaussian noise (std=0.2)
np.random.seed(42)
SENSOR_SEQ = 60  # 1-hour lookback (60 minutes)
SENSOR_HORIZON = 1  # predict next minute
ANOMALY_THRESHOLD = 3.5  # mm/s^2 — bearing replacement recommended above this

# TODO: Normalise vibration data and build windowed dataset
# TODO: Train both GRU (hidden=16) and LSTM (hidden=16) on sensor data
# TODO: Evaluate both models and compute MAE
# TODO: Benchmark inference latency for sensor-sized models (n_runs=500)
# TODO: Calculate real-time capacity: 60_000 / latency_ms = inferences per minute

# TODO: Anomaly detection: count predictions exceeding ANOMALY_THRESHOLD
# TODO: Print comparison table: MAE, latency, capacity, anomaly counts

# TODO: Visualise (2-row figure, 14x8):
#   Top: Time series of actual vs GRU predicted vibration (last 1000 min)
#     - Add horizontal anomaly threshold line (red dotted)
#     - Fill anomaly zone in red
#   Bottom: Daily average vibration bar chart (last 7 days)
#     - Color green if below threshold, red if above
#   Save to OUTPUT_DIR / "03_gru_smrt_vibration.png"



### Checkpoint 7 (Apply)


In [ ]:
# assert gru_mae < 1.0, "GRU vibration MAE should be reasonable"
# assert abs(gru_mae - lstm_mae) < 0.5, "GRU and LSTM should have similar accuracy"
# assert (OUTPUT_DIR / "03_gru_smrt_vibration.png").exists()
# print("--- Checkpoint 7 passed --- SMRT predictive maintenance application complete\n")



## REFLECTION


[x] Built GRU regressor ({n_params_gru:,} params, {param_ratio:.0f}% of LSTM)
  [x] Head-to-head: GRU val={gru_results['final_val_loss']:.4f} vs LSTM val={lstm_results['final_val_loss']:.4f}
  [x] Latency: GRU is {speedup:.2f}x faster than LSTM on stock data
  [x] Hidden state dynamics: GRU's update gate creates sharper transitions
  [x] Applied GRU to SMRT predictive maintenance (vibration monitoring)

  Key insight: GRU achieves similar accuracy to LSTM with 25% fewer
  parameters and measurably lower latency. The tradeoff matters most in
  real-time systems (sensor monitoring, trading) where you need to serve
  thousands of predictions per second. For offline batch processing where
  latency doesn't matter, LSTM and GRU are interchangeable — pick either
  and spend your time on feature engineering instead.

  Next: 04_temporal_attention.py — letting the model decide which past
  timesteps matter most.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)



## DIAGNOSTIC CHECKPOINT — GRU (3 gates vs LSTM's 4)


## DIAGNOSTIC CHECKPOINT — GRU (3 gates vs LSTM's 4)


In [ ]:
print("\n── Diagnostic Report (GRU) ──")
diag, findings = diagnose_regressor(
    gru_model,
    val_loader,
    title="GRU",
    n_batches=8,
    train_losses=gru_results["train_losses"],
    val_losses=gru_results.get("val_losses"),
    show=False,
)
# ══════ EXPECTED OUTPUT (synthesized reference — full run produces similar pattern) ══════



## DL Diagnostics Report — Prescription Pad


In [ ]:
#   [✓] Gradient flow (HEALTHY): min RMS = 2.7e-04 at
#       'gru.weight_hh_l0'. Comparable to LSTM (02) despite
#       25% fewer parameters (3 gates vs 4).
#   [✓] Saturation   (HEALTHY): reset gate mean activation
#       0.48, update gate mean 0.52 — both in the active
#       [0.2, 0.8] range, no stuck gates.
#   [✓] Loss trend    (HEALTHY): train slope -3.1e-03/epoch,
#       val slope -2.6e-03/epoch. Train-val gap 7%.



## Final val loss: ~1.3 after 15 epochs, sequence_length=60.

STUDENT INTERPRETATION GUIDE — reading the Prescription Pad:

 [BLOOD TEST — GRU vs LSTM EQUIVALENCE] RMS 2.7e-04 at
    gru.weight_hh_l0 is within 10% of LSTM (02). Cho et al.
    2014 (Slide 5O) demonstrated that GRU's simplified
    gating (reset + update merged into one cell, no
    separate forget/output split) preserves the additive
    gradient highway without the 4-gate overhead.
    Parameter-for-parameter, GRU often matches LSTM on
    tasks <500 timesteps.
    >> Prescription: Prefer GRU when compute/memory-
       constrained (mobile, edge). Prefer LSTM when task
       has VERY long dependencies (music generation,
       document summarisation) where the explicit cell
       state matters.

 [X-RAY — 3-GATE SATURATION] weight_hh_l0 packs THREE
    gate matrices (reset, update, new-candidate) into
    shape [3*hidden, hidden]. Healthy GRU shows both
    reset and update gates actively modulating — neither
    stuck at 0 (always forget) nor stuck at 1 (never
    forget). The "new-candidate" tanh should stay below
    0.95 max absolute.
    >> Prescription: If update gate saturates high
       (>0.9 mean), the GRU stops learning new info —
       indicator that task doesn't need recurrence at
       all, or that LR is too low. Increase LR or check
       whether a feedforward net suffices.

 [STETHOSCOPE] 7% train-val gap is slightly better than
    LSTM's 10% on the same PM2.5 task. GRU's fewer
    parameters = less capacity to overfit. For small
    datasets (<10k sequences), this advantage
    compounds.
    >> Prescription: For tiny datasets, prefer GRU. For
       large datasets where every drop of capacity
       matters, prefer LSTM.

 FIVE-INSTRUMENT TAKEAWAY: GRU diagnostics mirror LSTM
 diagnostics — same healthy gradient pattern through
 time, same gate-saturation checks (one fewer gate to
 read). This forward-references 04_temporal_attention
 where a fundamentally different mechanism (attention)
 tackles the same long-range-dependency problem, and
 05_architecture_comparison which puts all four
 architectures on the same diagnostic scale.


### Checkpoint 3


In [ ]:
assert len(gru_results["train_losses"]) == EPOCHS
assert gru_results["final_val_loss"] < 5.0
print(f"\n  Final val loss: {gru_results['final_val_loss']:.4f}")
print("--- Checkpoint 3 passed --- GRU trained\n")



## TASK 4 — Train LSTM for head-to-head comparison


In [ ]:
print(f"\n== Training LSTM (comparison) on {PRIMARY} ==")
lstm_results = train_model(
    lstm_compare,
    "LSTM_compare",
    tracker,
    exp_name,
    train_loader,
    val_loader,
    device,
)

print(f"\n  == Head-to-Head Comparison ==")
print(f"  {'Metric':<25s} {'GRU':>10s} {'LSTM':>10s}")
print(f"  {'Parameters':.<25s} {n_params_gru:>10,d} {n_params_lstm:>10,d}")
print(
    f"  {'Final val loss':.<25s} {gru_results['final_val_loss']:>10.4f} {lstm_results['final_val_loss']:>10.4f}"
)
print(
    f"  {'Avg grad norm':.<25s} {np.mean(gru_results['gradient_norms']):>10.4f} {np.mean(lstm_results['gradient_norms']):>10.4f}"
)



### Checkpoint 4


In [ ]:
assert lstm_results["final_val_loss"] < 5.0
print("--- Checkpoint 4 passed --- head-to-head comparison complete\n")



## TASK 5 — Visualise: inference latency comparison


Measure average inference latency in milliseconds.


In [ ]:
# In production systems (real-time sensor monitoring, trading), inference
# latency matters as much as accuracy. GRU's fewer operations per step
# translate to measurable speed gains.


def benchmark_inference(model: nn.Module, name: str, n_runs: int = 100) -> float:
    model.eval()
    test_input = torch.randn(1, SEQ_LEN, N_FEATURES, device=device)

    # Warmup
    with torch.no_grad():
        for _ in range(10):
            model(test_input)

    # Timed runs
    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_runs):
            model(test_input)

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_ms = (time.perf_counter() - start) / n_runs * 1000
    print(f"  {name}: {elapsed_ms:.3f} ms/inference (avg over {n_runs} runs)")
    return elapsed_ms


print("\n== Inference Latency Comparison ==")
gru_latency = benchmark_inference(gru_model, "GRU")
lstm_latency = benchmark_inference(lstm_compare, "LSTM")
speedup = lstm_latency / max(gru_latency, 1e-6)
print(f"  GRU speedup: {speedup:.2f}x faster than LSTM")

# Visualise latency comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: latency
models_names = ["GRU", "LSTM"]
latencies = [gru_latency, lstm_latency]
colors = ["#4CAF50", "#2196F3"]
bars = ax1.bar(models_names, latencies, color=colors, width=0.5, edgecolor="white")
ax1.set_ylabel("Inference Latency (ms)")
ax1.set_title("Single-Sample Inference Latency")
for bar, val in zip(bars, latencies):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{val:.3f}ms",
        ha="center",
        va="bottom",
        fontweight="bold",
    )
ax1.grid(True, alpha=0.3, axis="y")

# Bar chart: parameters
params = [n_params_gru, n_params_lstm]
bars2 = ax2.bar(models_names, params, color=colors, width=0.5, edgecolor="white")
ax2.set_ylabel("Parameter Count")
ax2.set_title("Model Size Comparison")
for bar, val in zip(bars2, params):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 100,
        f"{val:,}",
        ha="center",
        va="bottom",
        fontweight="bold",
    )
ax2.grid(True, alpha=0.3, axis="y")

fig.suptitle(
    f"GRU vs LSTM: {param_ratio:.0f}% parameters, {speedup:.2f}x faster",
    fontsize=13,
    fontweight="bold",
)
fig.tight_layout()
fig.savefig(str(OUTPUT_DIR / "03_gru_latency_comparison.png"), dpi=150)
plt.close(fig)
print("  Saved: 03_gru_latency_comparison.png")



### Checkpoint 5


In [ ]:
assert gru_latency > 0 and lstm_latency > 0
assert (OUTPUT_DIR / "03_gru_latency_comparison.png").exists()
print("--- Checkpoint 5 passed --- latency comparison complete\n")



## TASK 6 — Visualise: GRU hidden state dynamics


Extract and visualise GRU gate activations using hooks.


In [ ]:
# GRU's update and reset gates control how information flows.
# Visualise how the hidden state evolves differently from LSTM.


def visualise_gru_gates(model: nn.Module, sample: torch.Tensor) -> None:
    model.eval()
    gate_data: dict[str, list[np.ndarray]] = {"update": [], "reset": [], "hidden": []}

    # Run step-by-step through the GRU to capture hidden states
    gru_layer = model.gru
    seq_len = sample.shape[1]
    h = torch.zeros(1, 1, HIDDEN_DIM, device=device)
    hidden_states = []

    with torch.no_grad():
        for t in range(seq_len):
            out, h = gru_layer(sample[:, t : t + 1, :], h)
            hidden_states.append(h.squeeze().cpu().numpy())

    hidden_matrix = np.stack(hidden_states)  # (seq_len, hidden_dim)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

    # Heatmap of hidden state evolution
    im = ax1.imshow(
        hidden_matrix.T,
        aspect="auto",
        cmap="RdBu_r",
        interpolation="nearest",
    )
    ax1.set_xlabel("Timestep")
    ax1.set_ylabel("Hidden Dimension")
    ax1.set_title("GRU Hidden State Evolution (all dimensions)")
    plt.colorbar(im, ax=ax1, label="Activation")

    # Compare activation patterns: GRU tends to have sharper transitions
    # due to the direct linear interpolation h = (1-z)*h_prev + z*h_new
    variances = np.var(hidden_matrix, axis=0)
    top5 = np.argsort(variances)[-5:][::-1]
    for dim in top5:
        ax2.plot(hidden_matrix[:, dim], label=f"Dim {dim}", linewidth=1.5, alpha=0.8)
    ax2.set_xlabel("Timestep")
    ax2.set_ylabel("Activation")
    ax2.set_title(
        "Top-5 Most Active Hidden Dimensions (sharper transitions = update gate)"
    )
    ax2.legend(loc="upper right")
    ax2.grid(True, alpha=0.3)

    fig.suptitle("GRU Hidden State Dynamics", fontsize=14, fontweight="bold")
    fig.tight_layout()
    fig.savefig(str(OUTPUT_DIR / "03_gru_hidden_dynamics.png"), dpi=150)
    plt.close(fig)
    print("  Saved: 03_gru_hidden_dynamics.png")


sample_input = X_val_t[:1]
visualise_gru_gates(gru_model, sample_input)



## TASK 7 — Visualise: predicted vs actual + training curves


In [ ]:
viz = get_visualizer()
plot_training_curves(viz, gru_results, "GRU", "03_gru")

preds_denorm, actual_denorm, _ = plot_predictions(
    viz, gru_model, X_val_t, y_val_t, norm_mean, norm_std, "03_gru"
)

plot_time_series_overlay(
    preds_denorm,
    actual_denorm,
    "03_gru",
    title=f"GRU: Predicted vs Actual Close ({PRIMARY})",
)

rmses = plot_horizon_error(preds_denorm, actual_denorm, "GRU")



### Checkpoint 6


In [ ]:
assert (OUTPUT_DIR / "03_gru_training_curves.html").exists()
assert (OUTPUT_DIR / "03_gru_time_series_overlay.png").exists()
assert (OUTPUT_DIR / "03_gru_hidden_dynamics.png").exists()
print("--- Checkpoint 6 passed --- GRU visualisations generated\n")



## TASK 8 — Register model


In [ ]:
register_best_model(
    gru_model,
    "GRU",
    gru_results["final_val_loss"],
    PRIMARY,
    registry,
    has_registry,
)



## APPLY — SMRT Predictive Maintenance: Real-Time Sensor Monitoring


In [ ]:
#
# BUSINESS SCENARIO:
#   You are a data engineer at SMRT Corporation, which operates
#   Singapore's MRT (Mass Rapid Transit) network carrying ~3.4 million
#   trips per day. Train wheels, bearings, and axles generate vibration
#   data captured by accelerometers at 1-second intervals.
#
# WHY GRU (NOT LSTM)?
#   Predictive maintenance on rail infrastructure requires REAL-TIME
#   inference — the model must predict the next minute's vibration
#   reading before the sensor captures it. With 200+ sensors per train
#   and 200+ trains running simultaneously:
#     - LSTM: ~{lstm_latency:.3f}ms per inference
#     - GRU:  ~{gru_latency:.3f}ms per inference ({speedup:.1f}x faster)
#   At 200 sensors x 200 trains x 60 readings/min = 2.4M inferences/min.
#   The {speedup:.1f}x speedup means SMRT can run prediction on 40K sensors
#   that LSTM cannot serve within the 1-second window.
#
# DELIVERABLES:
#   - Next-minute vibration prediction with anomaly threshold
#   - Latency comparison: can GRU serve all sensors in real-time?
#   - Maintenance alert: "bearing X shows increasing vibration trend"
print("\n" + "=" * 70)
print("  APPLY: SMRT Predictive Maintenance — Vibration Monitoring")
print("=" * 70)

# Generate realistic vibration sensor data
np.random.seed(42)
n_readings = 60 * 24 * 30  # 30 days at 1-minute intervals
base_vibration = 2.5 + 0.3 * np.sin(2 * np.pi * np.arange(n_readings) / (60 * 24))
# Add gradual bearing degradation in the last 5 days
degradation = np.zeros(n_readings)
degrade_start = n_readings - 60 * 24 * 5
degradation[degrade_start:] = np.linspace(0, 1.5, 60 * 24 * 5)
noise = np.random.normal(0, 0.2, n_readings)
vibration = (base_vibration + degradation + noise).astype(np.float32)

# Normalise
SENSOR_SEQ = 60  # 1-hour lookback (60 minutes)
SENSOR_HORIZON = 1  # predict next minute
split_idx = int(0.8 * n_readings)
v_mean = vibration[:split_idx].mean()
v_std = vibration[:split_idx].std() + 1e-8
v_norm = (vibration - v_mean) / v_std

n_w = n_readings - SENSOR_SEQ - SENSOR_HORIZON + 1
X_sensor = np.stack(
    [v_norm[i : i + SENSOR_SEQ].reshape(-1, 1) for i in range(n_w)]
).astype(np.float32)
y_sensor = np.stack(
    [v_norm[i + SENSOR_SEQ : i + SENSOR_SEQ + SENSOR_HORIZON] for i in range(n_w)]
).astype(np.float32)
sp = split_idx - SENSOR_SEQ

X_st = torch.from_numpy(X_sensor[:sp]).to(device)
y_st = torch.from_numpy(y_sensor[:sp]).to(device)
X_sv = torch.from_numpy(X_sensor[sp:]).to(device)
y_sv = torch.from_numpy(y_sensor[sp:]).to(device)

# Train a small GRU for sensor data (latency-optimised)
sensor_gru = GRURegressor(input_dim=1, hidden_dim=16, horizon=SENSOR_HORIZON).to(device)
sensor_lstm = LSTMRegressor(input_dim=1, hidden_dim=16, horizon=SENSOR_HORIZON).to(
    device
)

for model_s, name_s in [(sensor_gru, "GRU"), (sensor_lstm, "LSTM")]:
    opt = torch.optim.Adam(model_s.parameters(), lr=1e-3)
    for epoch in range(15):
        model_s.train()
        opt.zero_grad()
        loss = nn.functional.mse_loss(model_s(X_st), y_st)
        loss.backward()
        nn.utils.clip_grad_norm_(model_s.parameters(), max_norm=1.0)
        opt.step()

# Evaluate both
sensor_gru.eval()
sensor_lstm.eval()
with torch.no_grad():
    gru_preds = sensor_gru(X_sv).cpu().numpy() * v_std + v_mean
    lstm_preds = sensor_lstm(X_sv).cpu().numpy() * v_std + v_mean
    actual_vib = y_sv.cpu().numpy() * v_std + v_mean

gru_mae = float(np.mean(np.abs(gru_preds - actual_vib)))
lstm_mae = float(np.mean(np.abs(lstm_preds - actual_vib)))

# Latency comparison for sensor-sized models
sensor_gru_lat = benchmark_inference(sensor_gru, "Sensor GRU (h=16)", n_runs=500)
sensor_lstm_lat = benchmark_inference(sensor_lstm, "Sensor LSTM (h=16)", n_runs=500)
sensor_speedup = sensor_lstm_lat / max(sensor_gru_lat, 1e-6)

# Capacity calculation
sensors_per_train = 200
n_trains = 200
readings_per_min = 60
total_inferences_per_min = sensors_per_train * n_trains * readings_per_min
gru_capacity = 60_000 / max(sensor_gru_lat, 1e-6)  # inferences per minute (single core)
lstm_capacity = 60_000 / max(sensor_lstm_lat, 1e-6)

# Anomaly detection: flag if predicted vibration exceeds threshold
ANOMALY_THRESHOLD = 3.5  # mm/s^2 — bearing replacement recommended above this
n_alerts_gru = int(np.sum(gru_preds.flatten() > ANOMALY_THRESHOLD))
pct_anomalous = n_alerts_gru / len(gru_preds) * 100

print(f"\n  Vibration Prediction Accuracy:")
print(f"    GRU MAE:  {gru_mae:.4f} mm/s^2")
print(f"    LSTM MAE: {lstm_mae:.4f} mm/s^2")
print(f"    Accuracy difference: {abs(gru_mae - lstm_mae):.4f} mm/s^2 (negligible)")
print(f"\n  Real-Time Capacity (single core):")
print(f"    Total demand: {total_inferences_per_min:,} inferences/min")
print(f"    GRU capacity:  {gru_capacity:,.0f} inferences/min")
print(f"    LSTM capacity: {lstm_capacity:,.0f} inferences/min")
print(f"    GRU speedup: {sensor_speedup:.2f}x")
print(f"\n  Anomaly Detection:")
print(f"    Threshold: {ANOMALY_THRESHOLD} mm/s^2")
print(
    f"    Alerts triggered: {n_alerts_gru} ({pct_anomalous:.1f}% of validation window)"
)
print(f"    These correspond to the last ~5 days where bearing degradation occurs")
print(f"\n  Business Decision: GRU achieves similar accuracy to LSTM with")
print(
    f"  {sensor_speedup:.1f}x lower latency — critical for SMRT's real-time monitoring"
)
print(f"  of {sensors_per_train * n_trains:,} sensors across {n_trains} trains.")

# Visualise sensor scenario
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Time series: actual vs predicted vibration (last 1000 minutes)
n_show = min(1000, len(actual_vib))
ax1.plot(
    range(n_show),
    actual_vib[-n_show:, 0],
    label="Actual Vibration",
    color="#2196F3",
    linewidth=1,
    alpha=0.7,
)
ax1.plot(
    range(n_show),
    gru_preds[-n_show:, 0],
    label="GRU Predicted",
    color="#4CAF50",
    linewidth=1,
    linestyle="--",
    alpha=0.8,
)
ax1.axhline(
    y=ANOMALY_THRESHOLD,
    color="#F44336",
    linestyle=":",
    linewidth=2,
    label=f"Alert Threshold ({ANOMALY_THRESHOLD} mm/s^2)",
)
ax1.fill_between(
    range(n_show),
    ANOMALY_THRESHOLD,
    actual_vib[-n_show:, 0].clip(min=ANOMALY_THRESHOLD),
    alpha=0.2,
    color="#F44336",
    label="Anomaly Zone",
)
ax1.set_xlabel("Time (minutes)")
ax1.set_ylabel("Vibration (mm/s^2)")
ax1.set_title("SMRT Train Bearing Vibration: GRU Prediction vs Actual")
ax1.legend(loc="upper left")
ax1.grid(True, alpha=0.3)

# Degradation detection
degrade_window = min(60 * 24 * 7, len(actual_vib))  # last 7 days
vib_last_week = actual_vib[-degrade_window:, 0]
daily_avg = [
    np.mean(vib_last_week[i * 60 * 24 : (i + 1) * 60 * 24])
    for i in range(degrade_window // (60 * 24))
]
ax2.bar(
    range(len(daily_avg)),
    daily_avg,
    color=["#4CAF50" if v < ANOMALY_THRESHOLD else "#F44336" for v in daily_avg],
    edgecolor="white",
)
ax2.axhline(y=ANOMALY_THRESHOLD, color="#F44336", linestyle=":", linewidth=2)
ax2.set_xlabel("Day (last 7 days)")
ax2.set_ylabel("Daily Avg Vibration (mm/s^2)")
ax2.set_title("Bearing Degradation Trend — Maintenance Alert")
ax2.grid(True, alpha=0.3, axis="y")

fig.tight_layout()
fig.savefig(str(OUTPUT_DIR / "03_gru_smrt_vibration.png"), dpi=150)
plt.close(fig)
print("  Saved: 03_gru_smrt_vibration.png")



### Checkpoint 7 (Apply)


In [ ]:
assert gru_mae < 1.0, "GRU vibration MAE should be reasonable"
assert abs(gru_mae - lstm_mae) < 0.5, "GRU and LSTM should have similar accuracy"
assert (OUTPUT_DIR / "03_gru_smrt_vibration.png").exists()
print("--- Checkpoint 7 passed --- SMRT predictive maintenance application complete\n")



## REFLECTION


[x] Built GRU regressor ({n_params_gru:,} params, {param_ratio:.0f}% of LSTM)
  [x] Head-to-head: GRU val={gru_results['final_val_loss']:.4f} vs LSTM val={lstm_results['final_val_loss']:.4f}
  [x] Latency: GRU is {speedup:.2f}x faster than LSTM on stock data
  [x] Hidden state dynamics: GRU's update gate creates sharper transitions
  [x] Applied GRU to SMRT predictive maintenance (vibration monitoring)
  [x] Real-time capacity: GRU serves {sensor_speedup:.1f}x more sensors per second
  [x] Anomaly detection: {n_alerts_gru} alerts for bearing degradation

  Key insight: GRU achieves similar accuracy to LSTM with 25% fewer
  parameters and measurably lower latency. The tradeoff matters most in
  real-time systems (sensor monitoring, trading) where you need to serve
  thousands of predictions per second. For offline batch processing where
  latency doesn't matter, LSTM and GRU are interchangeable — pick either
  and spend your time on feature engineering instead.

  Next: 04_temporal_attention.py — letting the model decide which past
  timesteps matter most.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

